In [ ]:
#@title Lesson 5 of 26: Neural Networks, Generalization and Clustering
from IPython.display import HTML, display
display(HTML('<div style="background:linear-gradient(118deg, #0d0726 0%, #2a0b5e 38%, #8f0f6b 72%, #ff6a1a 100%);border-radius:22px;padding:38px 36px 30px;color:#fff;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;box-shadow:0 12px 40px rgba(60,10,90,.35)"><div style="font-size:12px;letter-spacing:.34em;font-weight:700;color:#ffd1ea">BASIC PRINCIPLES OF ARTIFICIAL INTELLIGENCE  ·  LESSON 5 OF 26</div><div style="font-size:42px;font-weight:800;line-height:1.08;margin:14px 0 14px;text-shadow:0 2px 18px rgba(255,80,160,.45)">Neural Networks, Generalization and Clustering</div><div style="font-size:16px;line-height:1.55;max-width:680px;color:#f5ecff">How models learn their own features and still work on new data: hidden units, overfitting, regularization, honest evaluation with held-out data, and K-means clustering.</div><div style="margin-top:22px"><a href="https://youtu.be/ty5CQDSOe38" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#fff;color:#4a0d66">▶ Watch the lesson</a><a href="https://www.youtube.com/playlist?list=PLDyCOTShSfCk" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:rgba(255,255,255,.16);color:#fff;border:1px solid rgba(255,255,255,.45)">All lessons</a><a href="https://colab.research.google.com/github/neural-modules/ai-principles-notebooks/blob/main/05_neural_networks_generalization_and_clustering.ipynb" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#ffb22e;color:#1a0d00">Open in Colab</a></div></div><div style="background:#141033;color:#ece8ff;border-radius:18px;padding:22px 26px;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;font-size:15px;line-height:1.5;border:1px solid #2c2466"><div style="font-size:12px;letter-spacing:.3em;font-weight:700;color:#a87bff;margin-bottom:6px">HOW THIS NOTEBOOK WORKS</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ff3d9a">1</span>Choose <b>File &gt; Save a copy in Drive</b> first, so your work is saved to your own account.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#a87bff">2</span>Run the set-up cell, then work down the page in order. Later exercises reuse earlier answers.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#2ad4ff">3</span>Replace each <code>...</code> with your answer and run the cell. The check runs straight away.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ffb22e">4</span>Stuck? Open the 💡 hint under the exercise. Some exercises have two.</div><div style="margin-top:14px;font-size:14px"><span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#11c48a;color:#03251a;padding:3px 10px;border-radius:999px;font-weight:700">✅ correct</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ff4d6d;color:#fff;padding:3px 10px;border-radius:999px;font-weight:700">❌ not yet, with a pointer</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ffb22e;color:#2a1700;padding:3px 10px;border-radius:999px;font-weight:700">⏳ still blank</span></div><div style="margin-top:16px;font-size:12px;color:#9d95c9">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</div></div>'))

In [ ]:
#@title Set-up: run this cell first
import base64, contextlib, html as _html, math, sys, zlib
import numpy as np

try:
    import torch
except ImportError:
    torch = None
try:
    import einops
except ImportError:
    get_ipython().run_line_magic("pip", "install -q einops")
    import einops


from IPython.display import HTML, display
import matplotlib.pyplot as plt
from cycler import cycler

PALETTE = {"pink": "#ff3d9a", "cyan": "#2ad4ff", "amber": "#ffb22e", "violet": "#a87bff",
           "mint": "#3dffb4", "orange": "#ff7a3d", "ink": "#ece8ff", "panel": "#141033", "night": "#0c0922"}
plt.rcParams.update({
    "figure.figsize": (6.8, 3.8), "figure.facecolor": "#0c0922", "axes.facecolor": "#141033",
    "savefig.facecolor": "#0c0922", "axes.edgecolor": "#3a3170", "axes.labelcolor": "#ece8ff",
    "axes.titlecolor": "#ffffff", "axes.titleweight": "bold", "axes.titlesize": 12,
    "text.color": "#ece8ff", "xtick.color": "#b9b2e6", "ytick.color": "#b9b2e6",
    "axes.grid": True, "grid.color": "#ffffff", "grid.alpha": 0.08, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "lines.linewidth": 2.4, "image.cmap": "plasma",
    "axes.prop_cycle": cycler(color=["#2ad4ff", "#ff3d9a", "#ffb22e", "#a87bff", "#3dffb4", "#ff7a3d"]),
})

_CARD = {"pass": ("linear-gradient(90deg,#0fae7a,#3dffb4)", "#03251a"),
         "wrong": ("linear-gradient(90deg,#e0245e,#ff6b6b)", "#ffffff"),
         "todo": ("linear-gradient(90deg,#ff9f1c,#ffd166)", "#2a1700")}

def _say(state: str, text: str):
    bg, fg = _CARD[state]
    try:
        display(HTML(f'<div style="background:{bg};color:{fg};padding:10px 16px;border-radius:12px;'
                     f'font-weight:600;font-family:sans-serif;margin:4px 0;max-width:860px">{text}</div>'))
    except Exception:
        print(text)


if sys.version_info < (3, 12):
    print("This notebook uses Python 3.12 syntax. Colab runs 3.12 or newer, so use Colab if a cell fails.")

class NotDone(Exception):
    """Raised when an answer is still the ... placeholder."""

_results: dict[str, str] = {}

def _ready(x):
    if x is ... or x is None:
        raise NotDone
    if isinstance(x, (tuple, list)):
        for v in x:
            _ready(v)
    return x

def _array(x):
    if torch is not None and isinstance(x, torch.Tensor):
        return x.detach().cpu().double().numpy()
    return np.asarray(x, dtype=float)

def close(answer, expected, tol: float = 1e-6, rel: float = 1e-6) -> bool:
    """True when answer matches expected to within tol (absolute) or rel (relative)."""
    _ready(answer)
    a, b = _array(answer), _array(expected)
    return a.shape == b.shape and bool(np.allclose(a, b, atol=tol, rtol=rel))

def same(answer, expected) -> bool:
    """Exact match, for counts, strings and choices."""
    _ready(answer)
    if isinstance(answer, str) and isinstance(expected, str):
        return answer.strip().lower() == expected.strip().lower()
    return answer == expected

@contextlib.contextmanager
def checking(label: str, praise: str = ""):
    try:
        yield
    except NotDone:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · replace the ... with your answer and run the cell again.")
    except NameError as e:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · run the earlier cells first ({_html.escape(str(e))}).")
    except AssertionError as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · {_html.escape(str(e)) or 'not quite yet. Open the hint above and try again.'}")
    except (TypeError, AttributeError) as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            _results[label] = "todo"
            _say("todo", f"⏳ <b>Exercise {label}</b> · your code still returns ... or None. Finish it and run the cell again.")
        else:
            _results[label] = "wrong"
            _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    except Exception as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    else:
        _results[label] = "pass"
        _say("pass", f"✅ <b>Exercise {label}</b> · correct." + (f" {_html.escape(praise)}" if praise else ""))

_CHECKS: dict = {}

def check(label: str):
    """Run the check for one exercise by hand, for example check("3.6")."""
    fn, praise = _CHECKS[label]
    with checking(label, praise):
        fn()

def _auto_check(result):
    import re
    m = re.match(r"\s*# Exercise (\d+\.\d+)\b", result.info.raw_cell or "")
    if not m or m[1] not in _CHECKS:
        return
    if result.error_in_exec is not None:
        _results[m[1]] = "wrong"
        _say("wrong", f"❌ <b>Exercise {m[1]}</b> · the cell raised an error (see above). Fix it and run the cell again.")
        return
    check(m[1])

_ip = get_ipython()
for _f in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_f, "__name__", "") == "_auto_check":
        _ip.events.unregister("post_run_cell", _f)
_ip.events.register("post_run_cell", _auto_check)

def progress():
    """One bar per part: how many of its exercises pass so far."""
    parts: dict[str, list[str]] = {}
    for label, state in _results.items():
        parts.setdefault(label.split(".")[0], []).append(state)
    if not parts:
        return _say("todo", "No exercises checked yet. Run an exercise cell to start.")
    rows = []
    for s, states in sorted(parts.items(), key=lambda kv: int(kv[0])):
        done, n = states.count("pass"), len(states)
        rows.append(f'<div style="margin:5px 0;font-family:sans-serif;color:#ece8ff"><span style="display:inline-block;width:64px">Part {s}</span>'
                    f'<span style="display:inline-block;width:320px;height:12px;background:#2a2357;border-radius:6px;vertical-align:middle">'
                    f'<span style="display:block;width:{320 * done // n}px;height:12px;border-radius:6px;'
                    f'background:linear-gradient(90deg,#ff3d9a,#ffb22e)"></span></span> &nbsp;{done} of {n} checked</div>')
    display(HTML('<div style="background:#141033;padding:14px 18px;border-radius:14px">' + "".join(rows) + "</div>"))

_say("pass", f"✅ Ready. Python {sys.version.split()[0]} · NumPy {np.__version__} · "
            f"PyTorch {torch.__version__ if torch is not None else 'missing'}")

# The answer checks. Each runs by itself after you run its exercise cell.
exec(zlib.decompress(base64.b64decode("""
eNqtfeuS48ax5n8/BQ5/7JAaNgWA99Zp+0ijkUZhWZrVjCzJjA4GSIIk3CBAAWBfZPul9hH2yTa/zKpCASz2jOxVaNgkLnXJzMpb
ZWYtf/7+h+XP3o2XHQdRUURP3cXC73v+bd9bBH0vwF9f/Q1wnb5sqqdjfLNN86jq/WGJBn5pNID3+t4V/zt7/MdmX8HAx2MD7vCK
f+HHLT35Sp78LS7yshvShb82Xr2qX6V7X9A9fzD+wx/+sIm33rKI01P3t971Hzz6r4irU5Hh1UP0mBxOh+5vNJOB39NPZ3HV/bnv
/RTcLH/seyv686rvPYQ3y7/SL/rzhWrob2oApQzh5+bcvP+hFryX9D4//IYelnH8rWcP42997w39T08/hHg61KOIUhqF6unnZ3vi
R0p65OfFYDAAVqgh9T24tTu7khGUPXog9D5RAyq9Ky/AJQtiq7islmmSxejsFzWMKNvJQOhGeYzWcZdAjnbo0jHpe8MJ/Y6zzTFP
surmqygtYxlcvt22Xrwa0uP08iiQJ95h+AQEeqasovVdd0Hf1oRp6rPX58tJxj9u5YVjEW+kzYd9XMTdd4vrvkf/f5dnMeZPXS7w
Xa7QrVvvj56vSLGBARprt8vN/deN98viWr/BLfUGJRGI3xscqPteTSLLhzjZ7auym0ZPcVE2SQuvRASWFRDqbfPCi4hyqCPvt+So
3uh78ncRXN/W7T7mxXJXRJuy+1MA0gPdgegsilvKEv0AdYEcbKLCNSDvSi3QT70uXifwxY/Hrrr4iVf2enRrJI+/EfjmpyouujsM
pUePdP8GQDYgKEMavKf+dkTMuzcKaETXcpGu8SUzz6qIkgyz7e77XhnHG/qs4mN5M/J90FAV3RAxaqjKOIoo2+SHAb0fndJqWWS7
Lt6UkRAwbrxikOXFIUq7gucugW7f631GQLJ5x17eIMicvbHHwyHzDp8fAu6WQBx1vou7PEY1KgYRsLQDmnbA047fvYxE8x6Plj6u
MFOCKbWjRrmyLq7oIo+SPszFh1ANcWVdVAi2CBoLhgBuEMtMTfAEztYY1iK89f6b5sxkuPzu+2/evWYQhOMaXadq3308Y6BYk2b9
099HbqEIZhcxFsyIO783MsZmJJATPiEAtyFBVK/8NLgT2rXQJQPld0gWfM4t8iOnLCG0HRpN0n27STz+oSbfv77cpC/jfN0c5+sP
NOqj2a9/+ObL9twHfggJNJ/1vfmUnvn287988eXnYIm4R6/GVyP+HPJnyJ8Bi0azog5l3D32vacmjlhAgBgOcZR1u5YMOfaIfKzf
T7TyP/nEC2tedMzTp2VcFF1anI9nLXOHwDs9dU8zVV+3SSUY6wseSVIRG3js4f1aGtNTJZrNZPH/jnX+eIlu7LEtrMHQyA35EnIK
N7oxzPZiz3q3Rhre82hvxmBOGMxNEKhBv21oIRZA1tQBkC3trtGuY+a3vWfQ9Vb++kCVpjJuUqOq33rj7eA+Kuh53HmrB7+dO1bu
PQGZuLpavJoIHlsqzJUAt0+DXxdxRMt9d/O+OMU1KpMNQSqNDoTjG4X3J/7yi6W6yBD45w8Crvgp7vLK/EHUS/Cb1ggJxVFKYiNP
70kJYTnyM8mmNM4Ek9QpDf4H+obFoCai2u178sKTeaHXUGzyLZFUVdF0SA5ny4qu9b1oTSirioTUlt9Nlg38dYvBKsnyQ4JlYTfe
rVuXzlnayiM9aKPdoCeIrMd7t83TTZfI5U4NJ9k8apJTZNogfeb8cbVJtttg06WH+3hjsb2+vrvt2d+ZLLc1ud/V5L6+B0pVf8QB
oEwsbo1ArIq+dx8xQcvoQCFjSyw+MN+sSYMJY1EVt4op4GstDNHBIDoeSXHsMlcBucgr99Et9OgH/R5+P7Ng0FINuTLZtXV+Ypis
+dCfl94hqvas/Vz9Vr+0OXTf9r1XenHT16bq/ba5Qvr2vVcOtVx1TGu5VixJFb3yXintFFrrrVrPLLKhnWrc//8bDOnSx6euUZ3v
SddJyoqak1Eo7D7skzT2sMJrZEZApgYLEd0OmnBQow/taPQp+u++5QlGjXn1evU7yZYH4SWll+UVj4GMi41hpMv41xMtHtKa8ZhF
WRZIaYKRzMLcfdXgxW8XNPQb7+7WcFHqFrezp67c6XkxmSg00rtbJuy7ejWAa7zq9W7/s3nyJG+8CFrRn/+ybMqKLkygnqipIf6C
16rfI/ydYFXR36n6TerBRP0e984saWr/lf9MB2cvvPp2+e03795jbdtdB2ooYWtoE/V7on5P1e+pGtJQDXHYGvKI+Ar6akxed94a
k1EQyAYsSFDeuVQExYh/4D/nPHiBZcMd9uv+ikEZHY5pbHV812t2DjV4cRXcPqMGbLdYBTSobVIQx24qAO41+QdNl0SO/NatvdKE
yrz/9u5qEn+lieztQi07MwsiyIGsPl6HJC9a+oP1pGbl+3h9twyWQVfb8GUZF5W3TnPitGwxWj6UENomq5zDwfgWWsTCb17qe50s
3kVVch+TDEhPcemt4nV+iGFbH/My4Tu0nksy6KKnzoU+r/wB6AxeF1741i1/MOU7+OwklVfu81O68R7y4o7aJXYELYRgl50Oq5iE
UZ53iL7evH7153eLF8EgeAFloltPvO+9eGEoS18NneAgNb+ywGF5lBQIWvd83OqJHk6DhemGFUhdYk4/xN/+6N3FZDF6SdX5j3rz
tSOsp4BGvfnQzpiDKnzQQsml0x39Kj3/w30qfDNexYHWGk+o+gylz03O/JpWyC6uvGofe6skKhvwD1vwD53wHzrhv19Ck2BT1Qnp
RcAEWK5z4hCegBuQ9kVRpfFkDIDGgIatARFTfPGeRq5aiYqYWkIrVzbesJxKnuFVAIM6pyF432fpE1/jlewBmEBAng3OZzhyzvDN
qG+b4npG3GW+9bRr50dST141ZjFqzYLaefFD/lB6MGSAG/uvZuF+b+B9kVd7L+KlQhPOYlLmDtGRlg1PJC+SXeIa/tg5/LI9/JCH
/2ZEY/4rjfmLBskRy427cIbkxDLZbKBn4/u4eKIhFNQvgPeSyK8x03FrpmOaKR7qe/J5ZT4H3lf5yUyLuLL6VcRr6jzZZaVjYpNL
lBe4KS+wKS/gNUe2CguJFUALIijPp80QX+JrPXVik0eQ20NC7wH6Rf7ADQnuGlCYtKBAusALRqVFtn5Nrmkc8ZKnMYBQQd/VQ97C
u2LUhoOym9YBoqkTRKzpK7zLLoCyvRkyvCroCS8SvrC1MNNxN7bcRUfF6ao8vWErDk3t4Zjch5rxsyro/fMRsH8M/+kxAbHPtwGw
aQtgpIG84CUbW+uUQJASzDHATRLtcrIw+7IO6KPwolVOwiupmJPSlaSAKxevyhsk0bI1za3I6aILcrMW5JgUSGIvD3j3LiYitTzr
2h0nTrqeBmN5iNKUHvLUSwTGU1Z5NLRCJg8/fCoKc77dkrHXAMSsBQhSwV58XoFCqMnANJqfKnCcUV/802nqzSb96cT3MLRSSKgk
0BOd0RTjqCpV5yV8t7t9ReR/yjZR8eSAw7z7Qdt5LIrLj2RLrmFP0r9VYXtky+S3+AYK5hCQaV4eOq/4g6HoVmQ+UGv0KV7xaPPU
XedpGh2J8podKl39sTzvedJnZ9gZ6dKz/8NdwLHOPcmlHwu6sC5gr97jKzWuEKr73jBsNV2T+bkWNkBch7Ub8AWgRUTY+ZpJo2rB
G2O+EYmW8sS3scEGxzn1/CPzZNx/pbmzWO8GsS9K3XWDgOYtApoTAfF8Ey1tuNWVusSM+Cdr6NRDYlr21kRamnSY4QhTElGr12Xs
HYir1aQFODnIKvDdUgl7ZX3ZMVMMSvGRGx5Q9xEc5RHMZR+2rvHWV5+3S66Y8YR4hrfCHEydaNeWZVg1zOhEnAlX1kpwx+y5NPa9
/AFsLP7Qe19fyzOy78W+jHK/K5IN9lx2vXq/apvQhQjen2Naka25OqY5GUl0WSh2hFbpQ1Pt4yA5ELE9dBkyX1NLIu5vOsQE46LT
9+LHKs6qG3tU5is9viZN4YbEVp7Ss/fEx26gI92T4XETmD7WOWlHp6L7td587Htf671HhRR0nRIJpOUNEzAtiLyg728///b1+/ev
Fx1quupgLxmIf0g21b68UdslmyJ6WCpB0o0eFbckrtlhDkUiDX4DITYiaBaPHTR0sKaFTSAaKgPV2pZkMAJAvYYF4bdNCP+iCuJF
uyghE5VFAnxuJJYy0QyZhyrqhrgrrZVH/DbPdpbk0dLojOjDM8uNKbHef1zARFdSeA6iHDERs8qK9vfJZkNqMW82YrP2pRfUN2nJ
Hk9V5wPND1XzAZhsZ0JNDJWyPURzz75OcJrKTi9aWF64w4tps/FOR8Up1HBhbORNpZsQbXGrsGXvhU57Lzyz90QwZyQL9YCag5vO
Rrxjgw8ZHl3xHnEJU8YNBgB+4ypfPAcEduKqvFjvlyzDex/q5O3TezytRUS0K2KlLj6JPkWNNGYftmYPa2s6749pRKydPkRPRIef
p4ecQAd2RSJfmC4gOp31RzSBeLMj3ZFJGFePySOtVDcFOehz6ALthszey5O1ZlztMUXZCsfg1PPnoCSuEDtaDBuNkcrraMqC17AF
LxiDs3l/QvDilQwVaTzv+4FSotE1+tBA4IVNeKDntlHhbWPio7oj7wEUQRB7YjnA7Bwb26R2R6cybhMyD5Q03xONRaNjpIDvALOx
J38jKSYCS6k2JNQfomLTdbkLCGI/0r9X9O+v9O8Ll0LzWyPURistasLa3KiUYclTuxKDOXAoKXutnui2LGvaWNKO90rhp+zYILZ6
JTZ9A3ejFu5GzpV+wXBdrqBuOa3XSuOZRbmeMImS0TkVVic4EbmtZbmPjlADuqO+7MUR6zW2aNigmMY8xq15wMCFmQJrkMhWvaec
a0djv4m+nynHhRdH6/2AVBmyjZT0o0GnWO9eccrY1sEoqYk0IXUfOpmDqiYX3YLLMs2PcbcdYOXbXiHLOajRHaBntm2jA6w7/RPa
qUfWb8XTUEaiTKwBnEkLOBMnkt0W6m6JgIYm+TFyMRNho4TaI7wDRIUPIdjgvuNq5zfdjPbHde69KjmodSrtEapA25+JeRl4+4jm
ZZxxShFxNY7oi0Wtwze64r4el4nqjUayuLv9zHtkR5/PdikYEEglKZmhc0xLA4TTFghhCVNDWn0nbfK67cR6iMAxt6Z9zdHg5Yvq
+fpaSFBT8SM71B00NXMiR8Tg7oOT/ymoBgie8aJtRURfDlbERpm/9RqznLVmyWbuqcrlXQjO0pKce7aTonR9IjpM8mzgfYWZwItA
FNr3tOBdR5m3EbGn+yWqLUsxkjOyfknTBijUmnNM/yNsX6Xa0nKIDmDk3f/I6FWtkQ5NArKIt7VkqAORPpG+0Ojy/Cq/n0WguBtv
0SEkkQbd2a3kz0Mov8JOvQWcHfqNmDZ+uV+Pwdqya1AB3qndPfBXbjv/yA7/IqDHshfIZvFnHkOzudrwgxopmYPgB4uWQ1TeNQhj
3iKMuZOD1KakglVZnegGa1il3lMBKP/x5prj+ayQtTe0TjY9RMExLMpNvVkU+j3ZWH+Da0DkrPevc+0oWVfNDgVo7COEr4fnHccb
6BQENo/7jo2dQ7pFQkt/T6s20q5UYqlwSbJ/SwnYD9mLYyYzbcetoqK76LTkFo1o1ryCJUtjJdnJY17MbrVBd2PMuWOS3eHBNKLx
3HR4pp1/p6MQWoDuzfzgLkm9rfLDzQfHIqalGQvJ0y0EwDap6hGVcbV84ge6Hdl51O6x0O+I6ZjGO974JWOXgTe7bEGGbQsyFAsy
mEiLnuoCzJVBI3yqCRDxj2SQ93x3NvBePxIZKA0UOzweFnECc3MTl2v8ZScKqwAk3wpoCEQfuzzfYLou09LYlopt1KzBismudcNW
TLa9WyQmsS/S0uVCVyHWZxxhQ6vvGa8W3xavFu9ACQBogUTiOtpuoWGI0aj0JIIrWIkBj0MIRynvl4Fm/FsEZtRObOVRsHzG4jSD
maId6RgeMyyRj3VHNg2cGaaB8lnUcvdFad4V3xEJ1XVFVoJIepqD2uQkuuHQU6IFWveYLeGaXdXKyQBNhhZ+JLQlIGDaGnRaWB9e
2ApW4Xxnm39n2Bw1twS7sH1HiGMKOxdblcDb4XljV65dRuUjWcVb0HNEs42Iw+0s8A5bZv/QafYPa7MfOGQ2jW2xITwlFwQU4vMW
m1se6OV4RiXAJiAh+caSbBOzwf6Pzb+useJAPBzqCnrBy40ZhK0ZwHSH/05MK5+ILd5ukzUThxijKTz4aJcZP/bxSCERCuEVX4Ia
WUUF2Aq1wXOu9g9rm/2j4SLSLy4KBk4jNtQGTw2b2RlE6HHqTdkDNEFukvCqhtkAzrAFHNjpn6/AlP3BKGBuMJGtXwKNbkc60EZQ
DOVNFHbmi9s0fkxWKX5t4tQBlNHvBwq2ci7A5LXA5PXvgUnf9wks2PrRIPkoMT6YIDgjtPcy6vDAYUPuPiLXg72iCxujjikzyx2d
idP1U5RZ4lTjsNPu5aXupQbRx3fS0h/QRlNYP1bJ+q6kfqgHDU9WVtX3ofV93rkVEW5LeUSEKV/sRpDw++X8sOWRGLJHokmlQaCo
dDqTNbw+FfdKpBN7r5qkq9xypOKBBGhopCfmhdZ9WQY56HbscsDxLqOgmOCgXMMP+wRbTmed+n/qXGgAH/z+0HofBEpoxpsEkwK/
/9RYveMWXODhmDMghgoKWD6KmSkvoxnVJqoio+Kv4gYkeDfSeNl5IbvY28TtMgdHnatwb0MdHkihpAX3AC7qYzhzlyCLHhGWlmzm
zS3ridnkY8Qqxy2bjISv/LTbK+Wkye0a4Jq0wAWfR0sA9A1EpJ9DQoNjLUf21FZPwuRILyJTBBuzBsED7xsCVUy3E2xBsoeXuQvc
63QJsd3paRNvHIA0fpa18JRaIhJ7W1wH/q3icer73KHg0SCWZMtkDrhFnsGCTAskTton+yDyB7jXG5zwrN1TJi23kxHWigMvAv+a
naA8RvVDieyRCSXylKjWL4i81m80UDVtoWpar3jNxWXAaLOMHtQ2Z6YWcH4q6XdpfGRE3SqqaB+nmyu0w+QPN8Sa94cjoXKNQMKq
A01uj8s2zfNCRGWoHKvhWDM9B0A5wQC8WsGzkdeiZZkmdyW4hDM9RKyLlnG61VNrSDELgLMWAGc8LoxQHIOKe/oT4hSv72PVWHEi
uEZVFRcZ/X0qzUY2U7DwFOLbVarWx4tSeG8iPiCYx9420sw0lHHTTQaRQ0Weu11YzE65A5I7l3WgWtCJCCaGc9vQATotFszPx+Kz
usiBmBdf6NxSNj6ic6Dm+Y4/pHI0FA6SmrjdNf31vQakOld55/frEWdNWtN3tvis0hA97jnkRkjt7F39annTucbm8cNNYBoSJsmU
0vIYlOsojbudNN91ag3jUWkYAtaOrXokh67kcIX/hq4xby0cuLVamu+WQy3gFJZF+eRJ2OJ7C+F4RO1drmlAK5iJcXwkDqR88jQi
B4ep3WXiKWgRIPLp5I+bAM+pmbi0xia12CRQoxRzVh/YAqfvoV32aSjQulUWurm8R1iXyoJUCQtwrxnmpaSOicNDOASBiF5LNuwe
VirhhUWYbwkuebKObaXf0d1zZkDHMEmLk0aVmZtt6vptWxfepC9lDqRy8jSiVmOakcLrQDYhO4Hor9aPlRrGs/AydigQgd/FZmvG
4t9n5DCqXQgOp7PKcSMyHyE5AFBoXTpDHMLg0VJPsgY6HDJBAhSbX4ZVcpzbGUKMP0G9X0f8sz/Vhjjv8alcY3yVhY34klJd2Ta3
6kYtL8PI6WUY1V6GtzUUEGqyhOJadvX0xejxe71zALwd8GZi3wOM5gy0zpCHA81uPvXQ3GVVyM4R+MgURD0Y2TjHftKdt+coFQD+
jpTWKD1FVYwtRw9tNOAStuASOuEybMBldY9Ox0gkwM7r+SwQSh523woSPzoBsiP+oVhtaVWl+Nc0j2uH3HIjnUudU8dVktaqD/cl
O7xBz4q95354vUp0qmnZ20MVIrW8FRc/ank0RuzROAfZqA2y+bMgu4+KBCGpDajZSaAd/YS3gT7N7JNk3+mQKQCxe4Te6jzT+MdC
JdmQCpaQUBQ8aFDodhrgGLXAMXKCY/wxLplV37snQlcA2/TcvprVPXtpunj6OY/M2Pc9WSNCNGAYXhA0Bj9uDX6sUgqgYlwz6g3/
DcO+AUCt3QYDT/Hveet53/dHjjeGU/PGUOWUcHwPxwo7GLSxgAU4hpoEOL9kVtq6MIiXF7Yq5/OeI2Na8SuHtXdYWxYE8aVfMofc
M1JKTGNwHB19syWbaM+smbSXCGkmLoaHQIKyz/U17pFxLzULwBKanfGqxkMani+VOdRoM1qVatzEZrhppGQBTVpOP+TGz9oIzUIY
gdgXnF4GMcVSpUEtkxa1TGwv0fBDcVp66J/V9o4XbTalZ1lNQkHycpZjPDsHTUzbKhyvlsV1ePtxKht6MQaIlDVZ9VWmXLxtGRna
t8ZDe05zM7zGNH0vGXhEuO6mDTKdzf6npos9y4+xXPD8//0/l+0Wa2YfY7YYZvkRAxTCfrZ1jrW2t2JPh5Yn9VlzhTfkJr/bWhm1
/CQj9pN8AVLYFPmRNysUufpQcLRW2ldpkQbDwunYUAFNQ2lVORmy/Sv7tAnyOqqHOM4cVO9MyTgt5f22lbDgnX6zLpy2jMhVeQHL
1I2PBgeYtaAxq/X34XW9MRFn7C8k4GzzVOepwLeiN4PZx+eY48eEnKiA7ccVZzo4Cprw7adVLRkeV5cLZvi+29M3X66w/p4pDvJI
PONpJQ7ppmupucrV/lm9L6KdluKyU/4646o7twb3cXoslyCyhveb3WkvSlFRNklpxdgyL+F47D810DdvoY90ss7ntaOqL7oVh0bB
0c9jKjiPQRlTlgvB2gWgQYggfKi5N1i/pn7BOvsDy31BbKIWBmcuq9F5boSKNIl2GXCja8yMbA7aEScsk1prEw2+OO9wIoDsaSaw
w4x52Xmul5CjAMKplr7cBYs4tFNv7BOuSbbv4+db8ye8iSL0kMV1YxiyaeuZJmr7PDjbqGzdsyz1xuA5KMBOTGlaiH7bRPQtZdAT
dzbhF831a3d3VFo6HrCsdTtISyS/mBS+Y5ETbyBOt5ZsE9KbjpxiAmvnEUFXZc62O4iNYaM1FbaDzrjF+ML+/zHOorR6smIAhrxB
P9LBGwFv0CtHcqACFLtzRL5Pep2PaVCFiIzrsErZ8B82tv1FJyRLonjqQ9PpExmmVQIX1QqJAYfVJrJQMG4Z6WM20uvsNQ4FH0qC
78hTSQbEYkGokuXnqZFC2wqRzPR5HcUpgZAc8C1b7lgRHHZl57o7gGx8Al9ZgdpxVJ1oRPN2qDZxDWTbn/sFvjJ+gZD3rbRzBPY6
nCNJdjxVfYkWgFHn0pW/6nP9nQtdcsgZXvX+ruz/k96R4y0x7+/KwXSieYSPraDsccsTMHZ6AsbNeAMU7mFLjn2hIKwL2+umggz7
kKwfWk60wi+ENrx/0F8y5OoIwuDTTHn/dewgLZUfdM6szmKHKnHBwUT2gXQPQ5cL9nGxHuVuWsHF475tbzUxFNUQfRHtKspWRWqx
RAnDeQO+wxZ8h074ji7A92LVsEu1emrjWYyEVkEeuxhPT28T4aYUVJOb7AU2oHmwANJ7Bs9LoulyQUO4bXqE6xibsQPJ3abc6p/5
cKH5khVJskug3WtAd9SC7ujcqY5dU3aSR5uEWJCFxYH311ZnygPPmoCgM0CcCnvh/bnyuEiDbO86+IbxeFAfS8EBFFNdEq5PKvHT
jYJBeu3AgMZlT2NCgF6pxix8vm6+IP0ZbcztXedyTvWjLBzVaP4Nb7qJMejL+Fom5FrtaKud/ud1PhVdz9l4tWSft3fHPtU9WZyj
Y4JU5rf0gBlVg1TGLVIh8dH5Vq3mgQ9XwUUXPK0/+Y7M+0eCkto2PWVFvDsRfZI5tdE64cB7B1d0YCm87JpWsuOzOo1Ii631HrbJ
5kwnHNdeoAfi+LtmcaDQTm84l9G3zlDNdfS0RCBiVxrsKyeJKAcPPscWBagRuvPF1YKybA8itx5UucrujhMUNdW4VAdRd7kk4nym
6y6E/LZD/k9amJlw5j+9eW0y1uMjCruREsWcVgl10iZIU4Uc5eBZFW7I+0PQuY+nEtkziUuFMq4UIqwHVeBOGCTDcntKUy6KBsBY
LpeHGkAhERrh+BNVqK6Lqh+4rXaQa+A58PCwPETZCXtc9prRYS0aR/WCNHHCiu4cwU+uTjhjw9UHNJB3X3/p1b2kBc8LC1OAu+Rx
8CQauJq2cDVVuhoXq9AutUrHOm6SHSJcfxJ8SZPYyye7+0j6Jy0bEzOlp6MWBaGlSKqK2C0raxipA40zC40/35+jsmas/bp8rK41
eSSudbznynh9uzqeVTpqDDu5KWqFBIaE2ecpoC4hVjVq4/1sS+DeZzSC5u17Wwj3XJ7Tai9xYX2egWG4Vsw1P8N7qMf7JksuYuLJ
GzEzmHBKlaZTL7RzY6ys8iNTZNvbcrwnBdTqlx/kfvVdJWGSbBM/8oOcFVcngzwjaP4tP6ANnX8rYKEG3YfcfHrMdozCPccoWPC6
6M0rbzpXV+1IBbzIKU0tF58zTuH56AXG5O/2+41bnq4xe7qwvts4avjvsjgC//Cnipgi5NV6+1O2wYrmwPdmyIJ4CRtRg9Kq+Auh
hbG78B0B5JhIon2hvYm25u3ePpnUJrKUukT0SW3Glcc0qZZEk8k6BjvwfSuPxqJ73tauCq59kHXvI/WlintcIY8Tl8fyIRnm6ooE
gI8d6eR4nbCERr1/YkMcrapv0ix75vSuBecs8H6G3rVYW0VkVAIf6RNwNemIVDyQRscLhpCqcVepnXaHn3HYG5BAPZwqRjZDp7eY
jSXyroM0aFFdzCN13jlyWjLeb7NExqRl3k+ce/ATd4L/3/PVgh2JFs6xfGT1snfHyEbJr2BHCmtYOg7iLPzAFIrsdtb7HK4spZPA
lJYL7PMzmRvtPCeSY8kmVtniuYdV2zk3gc0E7m7ZY1fzC1hBL/5x968XnALq7Z8ImpzIF2PxSExHxw2LIj7mhSrQhPK6Kkv1VunA
xt8lN+XpBjLCFjJg7r+W5BuJJdkhLUbJcGvRE/SUSwscG4kKnD8DTYK3q/rPhR9NLpSL06V8h6qmgI9AeC7RDHV0+dxtPVFWhXhR
SGleLVo47hgblNAXyTJ1rIc0ju6wfE0/Y72g/cGMCxOgK4nloEVd1xvGEleLj37NfK2B4h3JALBqEHvTBvyHLfjbeRGzqQZxdFC4
K73ZtG5ehxCLNUX/aMGRBJ/5Doi7y9eJ2lUeEIFoCik7Z81qzkzyI7FY3pEE++a7r99x8U3mBep9yPpTaR5mgFtZTwpqHy/TWxK9
0S8TAmlc9TSe2ZL7CBlqpKUaJTAnETOVReqd8/QDwQ49KpPn+Nau2Hu9zmX5Omk5LCZ2xoH4hlEfQEZjxdPOBmMdloyHCEkD7zUA
rf3G9nBld5ksLdKfU5U7kyJLUa1qM3bmHQ7SuVCBYcNUwkWXy1+LqksGwwQqsJJ+VnGcIs958YEmHtlG+/RMHqpqE1G6FRk6mE8I
sRdb5/so37ZBRvcmvk+YL5WNxTVuAXdsLS5/OFSZHP5kdA1bvziRlKhXr1pVnKB3Ksk6o7viwVB7k95UUrtmEwfMLmQsZMQBEK/G
81rHCVmVPNFP+NQVNUcJGuFpIq5DpqrMoUfv2IXxfOzJg6G+05csgRju8AYQJi0gcDJCf4wAlPe5tz4pYZqvsGNJJjRxoKf85GGk
2uflW7USakZXJtjs4JdR4MnsaRkztYX9zAGmae3nhKvaqGR2GXJ3eCG/0JOCymzAHqOkYB867jiMFqZxjupMeiYggYueS9+4lOCn
KHZJaW3S9+pCl1rLQoJmTBoYAKSzPPOsJaiJaBo63v/SOl4P5ar9VvF1PaVOxF9flLWyokwlLX1FBzycJDyk1vEstE9baJ86tazZ
ZT++8jA3fMuNtbq+h4tNu3h1jflnXfi1hBapPJYSk2J5nuOMelixM/HcY0rd9T7gr1yTcV9enRuTFpBmLSDBunnVfu2I1DTlGGN2
DC9JWRU5EUchCTrnypG3STZSRU+0z5nm10yhqEgpBfIOuYT3SBZEvt1ydo8E8EopoY1ETRN+4Pd0rKG5i9Uc4/gOMrnTCE0W/rU7
JeATkVbxCBnaG9mAzrwFnTnXtcEGp92aeCvLWhHpq0CnqAESli3eX5DhCn0GmrnazanlFCLcT1mVpNr9oARfUooC6zLsnLvkd2TM
LTpKT7ftBNLxiozsEfFSWM9K4TSJ8mw8CEenMi51ZJbKEzeZ3mdmRG0/CK2QLoMfFyyIC6YCj4tthaZB0LIXALiV9h8XJVQVzTQM
Rprml9+2v7DB/dP5HMG7FRiMBmQZT5zOUzwJ9XLZfRrNGtUtuFiRhXt4X89QN72wbb2mZbJEhaM0Idtr02X/defNm/cd2bQODcOH
rtBXRjHLp87HtDZE0r5ubcpCd8ifU+2XPtZ6axUlNvCmLdt16rRdpxeKkfNY7tkQWPxKXeIfy/JfRR79ysTD84WCxFup8x6r1kfI
ffqnRT/ilDE+9p41Bhi2Bhiy7zzA5rjZvhjNtOYzC8TCgABZ59mWA349DJRk2jFFBKfgneAgWawOPLptOQJ633srpyMp9YZH/1bF
eysk1rhhzliHxbsCEJQ0UNCBH3Oi9mKF/KM7LVjaIVvTlqE1ZUOLDMjJVOoFbwsS4SBWrE5JWdR1rpJjCcbFJwva4+UKQgIsUD7q
vLjiNKdnppcw6CLfLENf7Fip/z4YNXfqcZd5X1RYdY5NNWE+j8Blx+a78nRQbXMbSo+mG10EDKEzvi5aHV1GHFEDWqMWtEbK7YdR
n9aV6JrsF7AqXGhzFA2qMstCc1fBbBjCWIfkLzbEygtd56Aud/Gwf9K1IpNVEekMW5NbiVZr6DvAbMyUtS+HRdrzDvWPY+Jwomdp
qlfmWgroo/SvL+U111IwDOzGK+p9ifRUqppnGeyQyrl5eSrLZYa9I1UVAx49ot6ZPq2mMTS2c+yB1nbUEWox6RT3tEKbdUdhE8K+
Y6Anu0PUwOO4hUfbApqzejcKuY2Qfg68z7mgm3Hg6amph+VSycKdoWHv03Au9ckho6e1LSSbZhcLcQQOzESrJdCidqfrV0Xb05UC
1AEJXS4txbsSsOGOlWsvklokjDS3wZiHDLEk9DYYO464NhiIIa/zKryvgdQE+bwAhfLDSsQHkymX5dahSq3wsmnLGpuyNWbqUnKE
mQkf76tKWWoNEbPSJQc0WhSLao4ItUSy3FOg4PpvSD/mUlz5vUmrceDJXYQP8WrIkBNPh9Ti60lRanuLqfGU2otmar+Sg5tA/BwE
QDSaJ8I9XyqNWpwmCZ9DQa913Au0DPr1ogka5z1J1kpH2sFKUGJAd0Y/gwYapi006G1L/YLQk5aY06EwNz1O6SOpHxjKYRW4THZj
slZlzaq6mhdG4AC5O9EbtQnOaunPe1byQiNpAdqA6MtQsvv1UQaMjLIx8Vlr4rB4/hYXuQqi05UGVH2axnTUxqwJpWsyZA6QEGko
Jc44BEtv5tZ4qPRJHQ5oGEvmKYcrAFxBHXP1GV3is3xe3nhjhy8KxL5c09rL+s/xl0W7hqRWF/AmlxC71DbhOU24gvFZ60+51TTi
EceNPW9JemErsIGJeQsTsK6+qgONeL1zXD3/fBlwxRGUFZUCEaqeWeCzhnJUhSBwSIaBsnAAXa02gg6R6nITOLuALmmujXfGOmY5
ORwJ34hgd2ntToMLBLIkult0dpoZSd7MbZ060qmXqJbnTDPWsNhqbEuUzjO9samyYbITs01RrLsvmxST8nzFNlV9v63rw05SXtaP
Wg58CFpJmtCRNUtBpA5HOQPt7IJBVP6KcyqscE9sejRjfHAsmJxbpCJvh001JXymSMRHNB/qkt7hWJofNZsf/p7mA65uPmwFKfHZ
I+rspZFKPmyeFJWgOPmGFNashJ+3dUzUrGWZzdgyC8TFy8aPUeihQuXX6jA27++5tnHg8y2Ymcoxaw4EGbvuy9pTiektSRIVyWOX
j4QjdsMntzn8ll+a6N+ZrjBs/EJkw4dqDC5D9ss+n9nX6kFl+iYq0lc4Q4Kq+Qo/5ogTaOqcK8o9NCAXtiDnividuauCs5eULhDX
OR/aQuWiCG7N/+rIMahHlQa6MbjbtXRmLbttZk57kqJ76n0+isWzj8cbiXJiDnRQUNZ1WuDsUAXBpTygtMCuOjkjj1+XY/QclDC6
YOGXFb0xmw38+qiQ0oRpyZAFXAd3HcM7Ll60xFtNgFrxSjN9IB3NiLPxz+gCBRD9W3V4obVLLxvy9QCkHr9wMRsfDQSMWgiAKThT
mugl0LLviOupQ6BpvGj7UCvHEEK4cICzAev6PnaA2r379IrLDo/hLBkNxgjYCvCdPm5vm3vBW5vwX5SXk+4Zeysceq/SF6CSN4AV
PzyzQk9HYuuxxoQbWzbPa9RMnuP7nItw9rg6Zn2FtYrMi0mKPGl6V4wRB/J5fN6Dha9xC18w+boaUJqsFawspKgS+CbcecbKB8HC
O1ttGbGckksZqI0wrZXuSH1EaI+OPRF+xNFADoNjdqFY+TpcorqzUPPbxfXInPfJ5SmHHG2KUy9eYpl+qgosNq5z+cpRAyyTFlhg
ePFKx6n01vJc5+KhqKAfkTJHQJjVlCvMgk9DpBvQCDjWYHSpssbsglF1d1ji+NEcKTF8vGVz/dZ17FWiisTkcVaBIUgeFV8T0oPd
C/dKXFxkLO1+mPYWU5DaWAgxFEeFtYokdMTQvQXSaQukU+YMWEDUBP1DqZIAHkYiDdGw16QuVFxOn0lENaoOQBU0hjopaQtTxByh
ZM44Y5vGAefZ5YiStZuazi0rPAintVVewnjbqrxCeE1LrjbgMWvBY2aRWN84EJoeLw4bZ/TJvhIKhcnuTxElim2DBhwzdu778LvL
09E6nw8Uwm2nUhKvFkuFTplBvn6aZ7Hr5BchOCh/HFofql0jKwmcWOzsvDTYrGXhzKTCke5cTRK5E4YJqSqnLIWpO4vWJSvuz1dS
20+5M9ZPRFUYBppy6dPGVHmF46kKVVMChzB/Q3YbfSwW+viSIT7GU9D9ORUR0sH/XxWLqLgdoKABaXHdb/7y9UAfGWFTEk5r0CPV
JeT4EBCdLxIf02gtBz5A7eMDSJTctYJiuHZ7IyxGVGcTHIPDduDNVWEtcUnLWR8QRWPjIBW5iEiVKqkQ7zKeTjg65lSUHeuJ6DEp
u518u+1YzQWmOQaAfjxoNDg6ay44b84V+zJrW1gzsbCsEeoDDkcquoXByOIn41rqEtenuYryKNciLTmgjIw+sJVaK/KIhCZQdkYt
84vbUcRAUGxJ8enWMcSLswOV5YBk90nE9YZ1WSlPaqZO5GFZYK2eecugmbNBM1GAYEEr1jLXtRa21K8lAlebKI1cUse6Nc02+4Jx
+R6SzSZ1gSe019Ky7+0LtZbOQTJtnRkduEByAdSraIO2bWgpAcT6iR7mKtpIZWyXHidSRZp6VTjlmLMlC/hhC/iwicZjJbZg11vW
IitTpHG0taE03lZKJRKShBhU3RMRFnqLVjRmftABd6fdRYNeIll4U7N5qR7DnmulHTJvr+eYpyeJglAbNnzipmzVN6Y+bE0dBtd3
ccKnmrEYgRcZM5KtX2TXaiNnPBaaTHPUDDokGbbM+lwmc7f3Jrh50bM0v2BTqaPD2SQq+9ZR4kMOr/T0dhYOyaH7TP2M3KGmfTko
fKPOi2mcNT7qNaY+ak3dVcNoXhskpkpae0QX9BD1oISzqIJyHStgpSZ0lxzm6u99Prpi7f3RtAGNlwRPXRBMNwx4qJJWHBYmZ6Hq
WNjGvMetecNkmKqzDbiZXa5PNmCftsH3CDrEIdesJxgqrc22fIln/ZlYsKZ2HYhbQmgjJhntc+K7y0aYX7ARyLakxgimrLRrHjSU
E5NxrLt9hY05LqNPo9entithzCr+no1VblDZLLKdkxxcpW1kM04TpH06fbPjpHkCfWJVHQllW1/K4hnuJqFBXCo6Wd81MDRpYQjW
y0ToXJdc0u8NvHc8QC4+q06mQGiLxbUSVTiMeU5trnFQqEQoOhAxbZ2vwiS9elreNQ5XubuWYC2zKO6gW/XlRL07dZbKnQUKxBb/
6xzIC9P84u7W+ZI+t+TS7dbiQtEA39PDqpXXu99Tk70ViyxBgrw1qb8O5Jgv8bs8U8CnDkZecxnc7mKoJzTEoSflTTDxLyYd/cZm
3s3QWfmnDjPS5hZN8jxkmRciM5vLOtq8ZePNrfrIwZhofExQgSNVH3MwRI4OrXapoAIHgmXz8YJjoMd88onW20q2QViZCxyE5yz4
E6er/GF5p+qYs1O9yI+a81PHIznsA21e8PUs74LQBF/Y59myb4MkpS5afeYUm7dMvTmbend0RW0U1izSl9UVhAYOemvtQcNfH3CA
EA+OOc9V5BbuG+Cp0harONuI7Ygo7xjuGkTr4qwQPj1C6Rsm6XUflQ6QGtPx3ftGDnH3UQDCy+OxWSaJ1i/fh9vIdV+xAT7yuSFk
fZtBMHMgjonY/bq0ZzEQIS0r6t37HrvHG+xz2TyT6QJKUXOAhjhRGxU628O4/UzMO18Dci/UShQCOZjIA1t0KanoqFeb52l3farK
ZcTBp73zo+K1csjS46OTIEJj570iko/uaku2CccW3N1QtU7hVdTFx/CiJd16x1jaJ3ifeDKeqpuTHMv2QUvJoYs6FPBzPMNK5q01
M+cjlMam9DdwZ+U4jAO+o0bClcTgyECuXim6NS925W+WISoluOCQrW/0wZ5JKa4PfUR1WTNGjZIHRPOKX0+GQ+84d+kC331+L7uH
rFPheSMd0RIj7xNJ4rNCTh/iletRhP7MVX7DyKQhYJOXs4h4K8u+PHemUETHo1Rs7J6PAedtnfXmfeqNzX4FAMsuTT0+cXCip/pE
Zb9lnAIoUMKD+i1arGMEmK3v4qq89kahlRQVWOkEBApA/KBrZ50VPaC2namAVURWxKJD73duUdqN5jvnxJF5IwW58Vgox06FerLo
vJ0LZjdOoDSNo+0wcLUtT0nbI9M2XZW2bbCFbbDBrqRRXAMTGlqYg1eTh4HcAG1ee/aDFsT7FoxdVOuO0sxZCU/73k8/fP/d1+IZ
VWE6v8VXEvgM20GV5FUbWeJ+0FH79gSH7QkO5TiAkUi9VbJTLxODUk52Bpa0b46rrMmcpPpDJlnFCqpEKaPQRSZu+1FCrZc8WJXb
i/H0vbnkuDE12IhTwK09BLxh3PlAy6E0HJjcOZ9dH3zE3aReYvTbXiIM+on2AnFhG4FPE6qjNlRdhildvpCoBRe2YgljPoV31uNS
BX7zfELxcJvhsUXhqpie8MmQwURHrQ5DlYrUHPS4Peix5E2ezT+YNQBQaLOE2o0OYMvI8TvGOUmya1w0jCSYWIzEfvc5ZjK5kMzW
zmUbSULWeCa408lYTpZLbz/zcv1uE0CTNoDsOrLInecY7TBUaSRjQ5lqJ6E8mXIfiD6Gx0e2vKXwYPYkHh4pYyg1ixypatTvhbi/
JTYwpXjMDCHc+iznWTNKedy5+K5vvVwHsgWXGmrJmGkbPlMbPpMJcwuRJSCgYMwn94G5cDATUiFxIEj0kMalLjGwjRIu6jtqCFJZ
9od44H2VFxaDFT/GngMoSCHI4l2a7JzeKxrd7MM1O7W9hniUsS/nxK6SjFhhlHbHwoYk5d9nlkEKXLVB2bHGc/75g7560kGayWHJ
5Bk1wjXVZZ8HImnzBnB2uXU8wQQE78EuzohN48yVupD2sImzWRtnMwtnYajzDqZ9da4dy/5TGpHYRb7Ue7hPzZLWWLCDMSPoZoWJ
JJS0RRc63LtjS8Vb6vzL5mJF6mUoK7aZZzn+QJ4lzX3enjsU3PkMDlKT4iT7FqourE6ltP04ggdaF2qBK3d+FtfbGtAscDY1G7dj
36miOoP0kHvInjaa3j86nJ+04voFnX+J8EMwqY5V4UOW4/T4GaOe+9qb4PNd1Jp7ayuHh0Cz/wsSxNgONdL0AVkaCgs660zclBzm
xZ4yldquy+dyES4NKws0Dn9hEFzYx0FEwVKOgr94ZrzKAUujJzglxInO57LKAfZ70MO+IeWCtiIc8DbNS07ikc8r86nKyJmjqVWy
nDmtPirlaGtWK8HRylhSUgLXNMPL0xTFpDVN2U/5NuFoDF7bOG25bzbY2wA4V4qROsvts8rQU0V4RyrsgkMvmrAJ27AJ7WB3mSZN
+s1I0D33TI5byfyB+4k3NihCFyjcqi0yS5cHRFEzHN79b6DabN4GPpdJrgtHSDKjCaVuJyJ9yEgf4dTDkbUdq7dO62HkZKom2U2H
bVFad/FjRfLnpnvFBysG+DBfaVRreumG1LM8pWfvSVu+gY19f4gebwLTyTrPKqKZLspQ4F/dW0rzSMsbDvFgLyJ9b52hjLpDRA+c
vV3ehLZjQM7FZb+AEJA66paPXVeHViPMkKuNoJmDNY+eOBsBwu5XUVrGl70CRCXDNpVwCQxvJbpMrrYNUAA4h6ccG0zs3+RMRdj7
iDMzB/kiIfVJgiyUsyyVI+/tgr/1tiunYR5Jb4r1KfF8hJuDyEYt9zcBYaYYiirIiEU9kEJuOB9vYzm+QIE0SXkkywbv4l9POFqD
xLm5JktTThjv149ilXbtC+q5GVc9GWzyEy0jJflz0wcXxBi8+/rLrup9UK8ttJYWN7721px51xo1znKOY+NDpBGGb4axPWW8JKN0
UObbCjvK3Su5S2RdkknEhPNLj4SrGkP3/PbPNAUSrHH8Gy1J+i4G6GAVre9w2it6zAdcoNCVtVWjoP/RfZBCVpFIpU6y0wGpBc0T
DFmpIdGFPKC6Et5YJdKZzCCpuGfm/eSVPceeHcaX39VOQBONKkkhZqd2FZvqw00OOmqvDRh+73jGdUHKl9MB6VP0CQZC36U0Mv0c
eF+3DxSX7Tnt2D4/n/w8ieUuPiLZip4zKcUyeNcicRuguih7faIN/ZWY4Zoz97z/YrVZrkcpX+85grSwdc5qinWEktrM1hkQpTpT
tAnMcRuYdTJcS6tEywPvC669LbxOsTpvpPMuRmcp0YoF9euKxPpoenm1hI9fHeZh5QJc4jeTFr+RTYglT+88sty+u+iAS2ObKFCG
Fukt/5Q/9PlPtQ2vyO35liTgEW1xNlfnitlR0GOBz999qW05VgZS9DGtYmGgTVU9kFH+3ev3s+biXdS1S03wiDzAISQTkrWDpIoR
42ykOGMjbogoZMQgTEM31mtSxaRNFTojzxZ1grxa7lixoHxHRM5Pah+VFV6tZ6ndo75BfXKeSu9JBTGQcQ44Sni6gyjae7DRer3k
o3VcmZP6Xk0PzuWHIyB+UYowqpLI7/pg0AYcFNHLcT4XI6mtrmsCOusca1x37uxZQfhj+7Z6FQLz/ojEVTdlGP0+rjx1zE7drCCg
SSfTNp3U3og5yeK5BAnP57W39j3H1T+YMJSksohCM4uacRlOo3hlmauTH2GVIb+DU/TwuIsyjPfhwbPY6Bl+/8tA2bXlhTGJo3Dx
sHj1/Q/fvf5h+f1XQM1aRc1aB8+JtjCqN9rV2RdyfJI6mAwtWjH3NkBnbYDO6tJz/B6Ht5cejG2OsNZuPwJMpmLNlUkasV8HJiGD
hyAfVebsbQIdZFp+vLbtTV00TRW4MMWeEx2Zas76jVTheZ1IYAK1/h+5juuW
""")).decode(), globals())


This notebook builds a small neural network by hand, then asks whether a model will work on data it has never seen, and ends by finding groups in data that has no labels. It uses the same examples as the video: four corners of a square that no straight line can split, ten noisy points on a wave, and eight unlabelled points for K-means. By the end you'll compute the two hidden units that fold the square, and draw the line that works.

You'll work through the same ideas as the video, in code:

1. Hidden units
2. The network
3. Training error and test error
4. Bias and variance
5. Regularization
6. Honest evaluation
7. Losses from likelihoods
8. K-means by hand
9. K-means traps
10. Error by subgroup
11. The payoff

Every section starts with a short explanation and a worked example you can run. Then come exercises that build on each other. Work them in order.

In [ ]:
#@title Helpers and imports
import math
import random
import warnings

import matplotlib.pyplot as plt

warnings.simplefilter("ignore", getattr(np, "exceptions", np).RankWarning)
rng = np.random.default_rng(0)
torch.manual_seed(0)
CORNER_COLOURS = [PALETTE["amber"], PALETTE["amber"], PALETTE["cyan"], PALETTE["cyan"]]


def draw_corners(ax, pts, title="", lim=(-0.3, 1.3)):
    """The four corners: amber for +1, cyan (the video's blue) for -1."""
    pts = np.asarray(pts, dtype=float)
    ax.scatter(pts[:, 0], pts[:, 1], s=160, color=CORNER_COLOURS, edgecolor=PALETTE["night"], zorder=4)
    ax.set_xlim(*lim); ax.set_ylim(*lim); ax.set_aspect("equal")
    if title:
        ax.set_title(title, fontsize=10)


def draw_net(ax, layers, title=""):
    """A network drawn as a graph: one circle per unit, one edge per weight."""
    xs = np.linspace(0, 1, len(layers))
    pos = [[(x, y) for y in (np.linspace(0.15, 0.85, n) if n > 1 else [0.5])] for x, n in zip(xs, layers)]
    for a, b in zip(pos, pos[1:]):
        for p in a:
            for q in b:
                ax.plot([p[0], q[0]], [p[1], q[1]], color=PALETTE["violet"], lw=1.2, alpha=0.7)
    colours = [PALETTE["cyan"]] + [PALETTE["pink"]] * (len(layers) - 2) + [PALETTE["amber"]]
    for col, c in zip(pos, colours):
        ax.scatter([p[0] for p in col], [p[1] for p in col], s=420, color=c, edgecolor=PALETTE["ink"], zorder=3)
    ax.set_xlim(-0.15, 1.15); ax.set_ylim(0, 1); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)


def draw_fits(ax, coefs, colour, title=""):
    """Polynomial fits over the wave, with the ten training points."""
    g = np.linspace(0, 1, 400)
    ax.plot(g, np.sin(2 * np.pi * g), color=PALETTE["ink"], lw=1, ls=":")
    for c in coefs:
        ax.plot(g, np.polyval(c, g), color=colour, lw=1.6, alpha=0.85)
    ax.scatter(TR_X, TR_Y, s=45, color=PALETTE["amber"], zorder=4)
    ax.set_ylim(-2, 2)
    if title:
        ax.set_title(title, fontsize=10)


def draw_clusters(ax, P, C, a, title=""):
    """Points coloured by their centre, centres as large crosses."""
    cols = [PALETTE["cyan"], PALETTE["pink"], PALETTE["amber"], PALETTE["mint"], PALETTE["violet"], PALETTE["orange"]]
    P, C = np.asarray(P, dtype=float), np.asarray(C, dtype=float)
    for p, k in zip(P, a):
        ax.plot([p[0], C[k][0]], [p[1], C[k][1]], color=cols[k % 6], lw=0.8, alpha=0.5)
    ax.scatter(P[:, 0], P[:, 1], s=70, color=[cols[k % 6] for k in a], zorder=3)
    ax.scatter(C[:, 0], C[:, 1], s=260, marker="X", color=[cols[k % 6] for k in range(len(C))], edgecolor=PALETTE["ink"], zorder=4)
    ax.set_aspect("equal")
    if title:
        ax.set_title(title, fontsize=10)

Here are the four corners from the start of the video. Amber corners are labelled $+1$ and the other two, drawn cyan here (the video's blue), are $-1$. Three lines try to split them, and each one gets at least one corner wrong. On the right, the same corners are plotted by the values of two hidden units, and one line now does the job.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))
draw_corners(axes[0], _XOR_X, "Every line gets a corner wrong")
gx = np.linspace(-0.3, 1.3, 2)
for slope, icpt, colour in [(-1, 1.5, PALETTE["pink"]), (1, 0.5, PALETTE["violet"]), (0.2, 0.4, PALETTE["orange"])]:
    axes[0].plot(gx, slope * gx + icpt, color=colour, lw=1.6, ls="--")
_, H_demo, _ = _net(_XOR_X)
draw_corners(axes[1], H_demo + np.array([[0, 0], [0.04, 0.04], [0, 0], [0, 0]]), "Plotted by the two hidden values")
axes[1].plot(gx, 0.5 - gx, color=PALETTE["mint"], lw=2)
axes[1].set_xlabel("hidden unit 1"); axes[1].set_ylabel("hidden unit 2")
plt.tight_layout()
plt.show()



## <font color="#ff2e88">Hidden units</font>

Each corner has two features: $x_1$ across and $x_2$ up. The amber points at $(0, 0)$ and $(1, 1)$ have label $+1$. The cyan points at $(0, 1)$ and $(1, 0)$ have label $-1$. The cyan points are where exactly one feature is on, a pattern called **exclusive or**, or XOR for short.

A linear classifier scores a point as $w \cdot x + b$, and its boundary is the line where the score is 0. No such line splits these four points; section 1 ends with you checking that by search.

So we build new features out of the old ones. Start with a linear score, then pass it through the simplest bend there is: keep the score if it is positive, and replace it with 0 if it is negative. That bend is **ReLU**, short for rectified linear unit:
$$\text{ReLU}(z) = \max(0, z).$$
Drawn as a curve, it is flat at 0 on the left and a straight ramp on the right, with a corner at 0.

A linear score followed by ReLU is a **hidden unit**. It is called hidden because its value sits between the input and the output: a feature the model computes on the way to its answer.

The first hidden unit is $h_1 = \text{ReLU}(x_1 - x_2)$. It is positive only below the diagonal, and of the four corners only $(1, 0)$ lights up, with value 1. The second is the mirror image, $h_2 = \text{ReLU}(x_2 - x_1)$, and only $(0, 1)$ lights it up.

Plot each corner by $(h_1, h_2)$ in place of $(x_1, x_2)$. Both amber corners land at the origin, and the cyan ones at $(1, 0)$ and $(0, 1)$. In this new plane the line $h_1 + h_2 = \tfrac12$ splits them, and the output score
$$s = \tfrac12 - h_1 - h_2$$
is $+\tfrac12$ at both amber corners and $-\tfrac12$ at both cyan ones.

Back in the square, $h_1 + h_2 = |x_1 - x_2|$, the distance from the diagonal measured along either axis. That sum folds the square along its diagonal, and one threshold splits the colours.

The bend is what makes this work. Without ReLU, each hidden unit is a linear score, and a linear score of linear scores is still a linear score.

In code we store the hidden weights as a matrix `U` with one column per unit, so all the hidden scores of a point `x` are `x @ U + C`. In NumPy, ReLU is `np.maximum(z, 0)`, and in PyTorch it is `torch.relu(z)`. Both work on every entry of an array at once.

In [ ]:
XOR_X = np.array([[0, 0], [1, 1], [0, 1], [1, 0]], dtype=float)
XOR_Y = np.array([1, 1, -1, -1], dtype=float)
U = np.array([[1.0, -1.0], [-1.0, 1.0]])    # column k holds the weights of hidden unit k
C = np.zeros(2)                             # hidden biases
V = np.array([-1.0, -1.0])                  # output weights
B = 0.5                                     # output bias

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
z = np.linspace(-2, 2, 200)
axes[0].plot(z, np.maximum(z, 0), color=PALETTE["pink"])
axes[0].set_title("ReLU: flat, then a ramp"); axes[0].set_xlabel("z")
g = np.linspace(-0.2, 1.2, 141)
G1, G2 = np.meshgrid(g, g)
im = axes[1].imshow(np.maximum(G1 - G2, 0), origin="lower", extent=(-0.2, 1.2, -0.2, 1.2), cmap="plasma")
draw_corners(axes[1], XOR_X, "Hidden unit 1 over the square", lim=(-0.2, 1.2))
axes[1].grid(False)
axes[2].imshow(np.abs(G1 - G2), origin="lower", extent=(-0.2, 1.2, -0.2, 1.2), cmap="cool")
axes[2].contour(G1, G2, np.abs(G1 - G2), levels=[0.5], colors=[PALETTE["mint"]], linewidths=2)
draw_corners(axes[2], XOR_X, "h1 + h2 folds along the diagonal", lim=(-0.2, 1.2))
axes[2].grid(False)
plt.tight_layout()
plt.show()



<font size="4"><b>ReLU</b></font>

Write `relu(z)` that works on a number or a NumPy array, and returns a float or an array of floats.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

`np.maximum` compares element by element. Pass `0.0` to get floats back.

</details>

In [ ]:
# Exercise 1.1: running this cell checks your answer
def relu(z):
    return ...  # your code here



<font size="4"><b>One hidden unit</b></font>

Write `unit(w, b, x)`: the value of a hidden unit with weights `w` (an array of 2) and bias `b`, at the point `x`. Use your `relu`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

The linear score is `w @ x + b`.

</details>

In [ ]:
# Exercise 1.2: running this cell checks your answer
def unit(w: np.ndarray, b: float, x: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>By hand at (1, 0)</b></font>

Without code, give the two hidden values at the corner $(1, 0)$ as `h_10 = (h1, h2)`, using $h_1 = \text{ReLU}(x_1 - x_2)$ and $h_2 = \text{ReLU}(x_2 - x_1)$.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Work out each score first, then apply ReLU.

</details>

In [ ]:
# Exercise 1.3: running this cell checks your answer
h_10 = ...  # your answer



<font size="4"><b>The hidden table</b></font>

Make `H4`, a 4 by 2 array: the two hidden values for each corner in `XOR_X`, in order. Do it in one line of matrix code with `U` and `C`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Send all four corners through the hidden layer with one matrix product; each row of `XOR_X` is a corner.

</details>

In [ ]:
# Exercise 1.4: running this cell checks your answer
H4 = ...  # your answer



<font size="4"><b>The output scores</b></font>

Make `s4`, the output score $\tfrac12 - h_1 - h_2$ for each corner, as an array, and `xor_ok`, whether every corner gets the sign of its label.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Send `H4` through the output layer the same way. A corner is right when its label and score have the same sign.

</details>

In [ ]:
# Exercise 1.5: running this cell checks your answer
s4 = ...  # your answer
xor_ok = ...  # your answer



<font size="4"><b>Predict, then check</b></font>

Predict the two hidden values at $(1, 1)$ before running anything, and store them as `h_11`. Then store `amber_same`, whether $(0, 0)$ and $(1, 1)$ get the same hidden values.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Both scores at $(1, 1)$ are $1 - 1$.

</details>

In [ ]:
# Exercise 1.6: running this cell checks your answer
h_11 = ...  # your answer
amber_same = ...  # your answer



<font size="4"><b>The fold</b></font>

Write `fold(X)` that takes an array of points with shape `(n, 2)` and returns $h_1 + h_2$ for each one. Then compute `fold_gap`, the largest difference between `fold(PTS)` and $|x_1 - x_2|$ over the random points `PTS`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

The hidden values come from the same product as `H4`; add them up along each row. Then compare with each point's distance from the diagonal, $|x_1 - x_2|$.

</details>

In [ ]:
# Exercise 1.7: running this cell checks your answer
PTS = np.random.default_rng(1).uniform(-1, 2, size=(500, 2))


def fold(X: np.ndarray) -> np.ndarray:
    return ...  # your code here


fold_gap = ...  # your answer



<font size="4"><b>Search every line</b></font>

Lines have the form $\cos\theta\, x_1 + \sin\theta\, x_2 + c$. Search every angle in `ANGLES` and offset in `OFFSETS`, calling $+1$ the side with a positive score, and store `min_mistakes`, the fewest mistakes any of these lines makes on the four corners.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Score the 4 corners against every direction with one matrix product. An extra axis and broadcasting add every offset at once. Count the mistakes for each line and keep the smallest count.

</details>

In [ ]:
# Exercise 1.8: running this cell checks your answer
ANGLES = np.linspace(0, 2 * np.pi, 360, endpoint=False)
OFFSETS = np.linspace(-3, 3, 241)
min_mistakes = ...  # your answer



<font size="4"><b>Without the bend</b></font>

Remove ReLU. Then the score is $(xU + c)\cdot v + b$, which equals $x \cdot w_{eq} + b_{eq}$ for one weight vector and one bias. Write `collapse(U, c, v, b)` returning `(w_eq, b_eq)`, and store `flat = collapse(U, C, V, B)`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Multiply out the brackets and group the terms that multiply $x$. What is left over is the bias.

</details>

In [ ]:
# Exercise 1.9: running this cell checks your answer
def collapse(U: np.ndarray, c: np.ndarray, v: np.ndarray, b: float):
    return ...  # your code here


flat = ...  # your answer



<font size="4"><b>Another network that works</b></font>

A second network uses $h_1 = \text{ReLU}(x_1 + x_2)$ and $h_2 = \text{ReLU}(x_1 + x_2 - 1)$, with output $s = -h_1 + 2h_2 + \tfrac12$. Make `s_alt`, its score at each corner in `XOR_X`, and `alt_ok`, whether all four signs are right. A map of its score appears when the check passes.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Each column of `U_alt` holds one hidden unit's input weights and `C_alt` holds the two biases, as in `U` and `C`. Read the output weights and bias off the formula for $s$.

</details>

In [ ]:
# Exercise 1.10: running this cell checks your answer
U_alt = ...  # your answer
C_alt = ...  # your answer
s_alt = ...  # your answer
alt_ok = ...  # your answer



## <font color="#7b2ff7">The network</font>

A **neural network** is layers of hidden units. The inputs feed the first layer, where each unit computes a linear score of all the inputs and applies ReLU. The outputs of one layer are the inputs of the next, and the last layer produces the score.

Drawn as a graph, each circle is a unit and each edge carries one weight. Our network for the square has 2 inputs, 2 hidden units and 1 output.

To count the weights, take each layer in turn: one weight per edge, plus one bias per unit it feeds. Our network has 4 edge weights and 2 biases into the hidden layer, then 2 weights and 1 bias into the output: 9 in all. Sizes grow fast. A network that reads a 28 by 28 image (784 inputs), with 100 hidden units and 10 outputs, has 79,510.

The last layer usually has no ReLU. Its score is used as it is: as a prediction for regression, or through the sign and the margin for classification. Everything below the last layer is a learned feature extractor, and the last layer is a linear model on top of those features.

Running the network on an input is the **forward pass**. At $(0, 1)$ the hidden scores are $-1$ and $1$, ReLU gives $0$ and $1$, and the output is $-\tfrac12$: blue, which is right.

The forward pass is a computation graph, the same kind as in the lesson on gradients, with one new kind of node, the ReLU. So training needs nothing new. Pick a loss on the output score, the logistic loss for example, and the **backward pass** sends slopes back through the graph until every weight has its gradient.

ReLU passes slopes simply: where its input was positive the slope is 1, and the slope from above passes through unchanged. Where its input was negative the slope is 0, and nothing passes. A unit that outputs 0 on an input learns nothing from it.

Each update is the SGD step from before. Trained from small random weights on the four corners, a network with 2 hidden units got stuck short of a perfect fit in 16 of 20 starts. The loss of a network is not **convex**: it has flat stretches and valleys that are not the lowest. With 8 hidden units, all 20 starts found a perfect fit.

Stack more layers and each layer builds features out of the features below it. **Deep learning** is the name for training networks with many layers. In PyTorch a network like ours is `nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1))`, and `loss.backward()` runs the backward pass for you.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
draw_net(axes[0], [2, 2, 1], "2 inputs, 2 hidden units, 1 output: 9 weights")
draw_net(axes[1], [3, 5, 5, 2], "Two hidden layers")
plt.tight_layout()
plt.show()

net_demo = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.ReLU(), torch.nn.Linear(2, 1))
for name, p in net_demo.named_parameters():
    print(f"{name:10} shape {tuple(p.shape)}")
print("parameters in all:", sum(p.numel() for p in net_demo.parameters()))



<font size="4"><b>Count the weights</b></font>

Write `n_weights(layers)`, the number of weights, biases included, in a network whose layer sizes are listed in `layers`, inputs first. For our network, `n_weights([2, 2, 1])` is 9.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Count the edges and biases for each pair of neighbouring layers. `zip(layers, layers[1:])` gives the pairs.

</details>

In [ ]:
# Exercise 2.1: running this cell checks your answer
def n_weights(layers: list[int]) -> int:
    return ...  # your code here



<font size="4"><b>The image network</b></font>

Give `mnist_weights`, the count for layer sizes `[784, 100, 10]` from your `n_weights`, and `torch_count`, the same count read from a PyTorch model: build `nn.Sequential(nn.Linear(784, 100), nn.ReLU(), nn.Linear(100, 10))` and add up `p.numel()` over its parameters.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`p.numel()` counts the entries of one parameter tensor, and `model.parameters()` lists them all.

</details>

In [ ]:
# Exercise 2.2: running this cell checks your answer
mnist_weights = ...  # your answer
model_784 = ...  # your answer
torch_count = ...  # your answer



<font size="4"><b>Deep or wide</b></font>

Compare two networks for the same image: `deep`, with two hidden layers of 100 units (`[784, 100, 100, 10]`), and `wide`, with one hidden layer of 200 (`[784, 200, 10]`). Store both counts.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Your `n_weights` twice.

</details>

In [ ]:
# Exercise 2.3: running this cell checks your answer
deep = ...  # your answer
wide = ...  # your answer



<font size="4"><b>The forward pass</b></font>

Write `forward(x, W1, b1, w2, b2)` returning `(z, h, s)`: the hidden scores, the hidden values and the output score. Use the same shapes as `U`, `C`, `V` and `B`, so it works on one point or on a batch of rows.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Three steps: the hidden scores, your `relu`, then the output score. Return all 3.

</details>

In [ ]:
# Exercise 2.4: running this cell checks your answer
def forward(x, W1, b1, w2, b2):
    ...  # your code here



<font size="4"><b>All four at once</b></font>

Run `forward` on the whole of `XOR_X` in one call and store the output scores as `s_batch`. Store `batch_shape`, the shape of the hidden values array.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The rows of `XOR_X` are the four inputs, so the same matrix code works unchanged.

</details>

In [ ]:
# Exercise 2.5: running this cell checks your answer
_, h_batch, s_batch = ..., ..., ...  # your answer
batch_shape = ...  # your answer



<font size="4"><b>The slope of ReLU</b></font>

Write `relu_slope(z)`: 1.0 where `z` is above 0 and 0.0 elsewhere, for a NumPy array. Use 0 at exactly 0, as PyTorch does.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`(z > 0)` is an array of booleans; `.astype(float)` turns it into ones and zeros.

</details>

In [ ]:
# Exercise 2.6: running this cell checks your answer
def relu_slope(z: np.ndarray) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>The backward pass by hand</b></font>

At the input $(0, 1)$, give the slopes of the output score $s$ with respect to the output weights, `g_w2`; with respect to the two hidden scores, `g_z`; and with respect to the hidden weights, `g_W1` (a 2 by 2 array, same layout as `U`).

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Apply the chain rule one layer at a time. The output weights see $h$ directly, each hidden score passes through the slope of ReLU, and each first-layer weight also picks up its input. `np.outer` builds a matrix from 2 vectors.

</details>

In [ ]:
# Exercise 2.7: running this cell checks your answer
x01 = np.array([0.0, 1.0])
z01, h01, _ = ..., ..., ...  # your answer
g_w2 = ...  # your answer
g_z = ...  # your answer
g_W1 = ...  # your answer



<font size="4"><b>Check it with autograd</b></font>

Build `W1t`, `b1t`, `w2t` and `b2t` as float64 tensors with `requires_grad=True` from `U`, `C`, `V` and `B`. Run the forward pass on $(0, 1)$ with `torch.relu`, call `.backward()` on the score, and store the gradient of `W1t` as the NumPy array `torch_gW1`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`torch.tensor(U, requires_grad=True)` keeps float64. After `s.backward()`, the slope sits in `W1t.grad`; use `.numpy()` on it.

</details>

In [ ]:
# Exercise 2.8: running this cell checks your answer
torch_gW1 = ...  # your answer



<font size="4"><b>Gradients on all four corners</b></font>

Write `xor_grads(W1, b1, w2, b2)`: the gradients of the mean logistic loss over the four corners, $\frac14 \sum \log(1 + e^{-y s})$, returned as `(gW1, gb1, gw2, gb2)` in the same shapes as the inputs.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Run `forward` on all four corners. Find the slope of the mean loss for each corner's score first, then push it back through the output layer, the ReLU slope and the first layer, as in the single-input case but with matrices.

</details>

In [ ]:
# Exercise 2.9: running this cell checks your answer
def xor_grads(W1, b1, w2, b2):
    ...  # your code here



<font size="4"><b>Stuck or not</b></font>

Train with your `xor_grads`. For each seed in `range(20)`, set `r = np.random.default_rng(seed)`, `W1 = r.normal(0, 1, (2, H))`, `b1 = np.zeros(H)`, `w2 = r.normal(0, 1, H)` and `b2 = 0.0`, then take 4000 gradient steps of size 0.5. A start is stuck if any corner ends with a margin at or below 0. Store `stuck_counts`, a dictionary from `H` to the number of stuck starts, for `H` of 2 and 8. A bar chart appears when the check passes.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Write a helper `mistakes_after(H, seed)` that returns the number of corners with margin at or below 0 after training. Each step is `W1 = W1 - 0.5 * gW1`, and the same for the other three.

</details>

In [ ]:
# Exercise 2.10: running this cell checks your answer
stuck_counts = ...  # your answer



<font size="4"><b>A unit that never fires</b></font>

Take `W1 = [[-1, 1], [-1, 1]]`, `b1 = [-0.5, 0]`, `w2 = [1, 1]` and `b2 = 0`. The first unit's score is $-x_1 - x_2 - 0.5$, negative on every corner. Use your `xor_grads` and store `dead = (gW1[:, 0], gb1[0])`, the gradients of the first unit's weights and bias, and `alive`, the gradients `gW1[:, 1]` of the second unit.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Build the arrays with `np.array(..., dtype=float)`, call `xor_grads` once and slice the columns.

</details>

In [ ]:
# Exercise 2.11: running this cell checks your answer
gW1_d, gb1_d, _, _ = ..., ..., ..., ...  # your answer
dead = ...  # your answer
alive = ...  # your answer



## <font color="#00a383">Training error and test error</font>

What we want from a model is good predictions on new data: examples it never saw during training. A model that does this is said to **generalize**.

Take a small problem: 10 points evenly spaced from 0 to 1, each one a smooth wave $\sin(2\pi x)$ plus random noise with standard deviation 0.25. The wave is the pattern, and the noise is the part no model can predict.

Fit three models by least squares: a straight line, a gentle curve of degree 3, and a curve of degree 9. A polynomial of degree 9 has 10 coefficients, one per point, so it can pass through every point exactly.

**Training error** is the average loss on the examples the model was fit to; here, the mean squared error on the 10 points. **Test error** is the same average on fresh examples from the same source, held back during fitting. We draw 1,000 test points from the same wave with the same noise. They play the part of the future.

The test points must stay out of the fitting. If they had been among the training points, the degree 9 curve would have threaded them too, and its test error would look as perfect as its training error.

A model whose training error is much lower than its test error **overfits**: it fit the noise in these particular points. A model whose two errors are both high and close together **underfits**: it is too simple for the pattern.

There is also a floor. The noise has variance $0.25^2 = 0.0625$, so on average no model can get a test error below that.

In NumPy, `np.polyfit(x, y, d)` returns the least squares coefficients of a degree `d` polynomial, highest power first, and `np.polyval(coef, x)` evaluates it. We also draw 10 more points, `VA_X` and `VA_Y`, for choosing settings later.

In [ ]:
NOISE = 0.25


def truth(x):
    return np.sin(2 * np.pi * x)


rng_wave = np.random.default_rng(18)
TR_X = np.linspace(0, 1, 10)
TR_Y = truth(TR_X) + rng_wave.normal(0, NOISE, 10)
VA_X = rng_wave.uniform(0, 1, 10)
VA_Y = truth(VA_X) + rng_wave.normal(0, NOISE, 10)
TE_X = rng_wave.uniform(0, 1, 1000)
TE_Y = truth(TE_X) + rng_wave.normal(0, NOISE, 1000)
DEGREES = (1, 3, 9)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, d, colour in zip(axes, DEGREES, [PALETTE["cyan"], PALETTE["mint"], PALETTE["pink"]]):
    draw_fits(ax, [np.polyfit(TR_X, TR_Y, d)], colour, f"degree {d}")
plt.tight_layout()
plt.show()



<font size="4"><b>Mean squared error</b></font>

Write `mse(pred, y)`: the mean of the squared differences, as a float.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Square the errors and average them with `np.mean`.

</details>

In [ ]:
# Exercise 3.1: running this cell checks your answer
def mse(pred, y) -> float:
    return ...  # your code here



<font size="4"><b>Three fits</b></font>

Make `fits`, a dictionary from each degree in `DEGREES` to its coefficients fitted on the training points.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`np.polyfit(x, y, deg)` returns the coefficients for one degree. Collect them in a dictionary keyed by degree.

</details>

In [ ]:
# Exercise 3.2: running this cell checks your answer
fits = ...  # your answer



<font size="4"><b>Training error</b></font>

Make `train_err`, a dictionary from each degree to its training error.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`np.polyval` turns coefficients into predictions; your `mse` compares them with the targets.

</details>

In [ ]:
# Exercise 3.3: running this cell checks your answer
train_err = ...  # your answer



<font size="4"><b>Test error</b></font>

Make `test_err`, a dictionary from each degree to its error on the 1,000 test points. Bars of both errors appear when the check passes.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

The same as `train_err`, with `TE_X` and `TE_Y`.

</details>

In [ ]:
# Exercise 3.4: running this cell checks your answer
test_err = ...  # your answer



<font size="4"><b>Read the bars</b></font>

Store `best_train_deg`, the degree with the lowest training error, and `best_test_deg`, the degree with the lowest test error. Find them with code.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`min` with a `key` function picks the degree with the smallest error.

</details>

In [ ]:
# Exercise 3.5: running this cell checks your answer
best_train_deg = ...  # your answer
best_test_deg = ...  # your answer



<font size="4"><b>Through every point</b></font>

Store `n_coef9`, the number of coefficients in the degree 9 fit, and `max_resid9`, the largest absolute gap between that curve and the 10 training targets.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

The number of coefficients is the length of the fit. A residual is a prediction minus its target.

</details>

In [ ]:
# Exercise 3.6: running this cell checks your answer
n_coef9 = ...  # your answer
max_resid9 = ...  # your answer



<font size="4"><b>Test points in the training set</b></font>

Fit a degree 9 curve to the first 10 test points, `TE_X[:10]` and `TE_Y[:10]`. Store `err_seen`, its error on those same 10 points, and `err_unseen`, its error on the other 990.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Fit degree 9 on the first 10 test points only. Then score that fit on those points and on the rest.

</details>

In [ ]:
# Exercise 3.7: running this cell checks your answer
c_leak = ...  # your answer
err_seen = ...  # your answer
err_unseen = ...  # your answer



<font size="4"><b>The noise floor</b></font>

Store `floor`, the variance of the noise, and `truth_err`, the test error of the true wave itself, `truth(TE_X)`, on the test points.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

The variance is the standard deviation squared.

</details>

In [ ]:
# Exercise 3.8: running this cell checks your answer
floor = ...  # your answer
truth_err = ...  # your answer



<font size="4"><b>Every degree</b></font>

Make arrays `train_curve` and `test_curve`: the training and test errors for each degree from 0 to 9. The two curves appear when the check passes.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Fit inside a list comprehension over `range(10)`, once per degree.

</details>

In [ ]:
# Exercise 3.9: running this cell checks your answer
train_curve = ...  # your answer
test_curve = ...  # your answer



<font size="4"><b>Choose with other points</b></font>

Choosing the degree by the test error uses the test set to make a decision, which section 6 explains is a mistake. Choose it with the 10 separate points `VA_X`, `VA_Y`. Store `val_curve` for degrees 0 to 9, `best_deg_val`, the degree with the lowest value, and `test_of_choice`, that degree's test error.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`int(np.argmin(val_curve))` is the degree, since the curve starts at degree 0.

</details>

In [ ]:
# Exercise 3.10: running this cell checks your answer
val_curve = ...  # your answer
best_deg_val = ...  # your answer
test_of_choice = ...  # your answer



## <font color="#ff5f2e">Bias and variance</font>

Suppose we could collect a fresh set of 10 points, with the same wave and new noise, and fit again, many times over. The straight lines barely move, and every one of them misses the wave in the same way. The degree 9 curves fly apart: each one threads its own points, and between the points they disagree wildly.

Those are two kinds of error. **Bias** is the error that remains when you average the fits over many training sets: the model is too simple to follow the pattern. **Variance** is how much the fit changes from one training set to the next: the model follows the noise in each sample.

Measured on a grid of inputs $x$, with $\bar f(x)$ the average fit over the training sets:
$$\text{bias}^2 = \text{mean over } x \text{ of } \big(\bar f(x) - \text{truth}(x)\big)^2, \qquad \text{variance} = \text{mean over } x \text{ of the variance of } f(x) \text{ across fits}.$$

On average over training sets, the test error splits into three parts: squared bias, variance and the noise. Changing the model's flexibility to lower one of the first two usually raises the other. That is the **bias-variance trade-off**. Plot both against flexibility, and their sum makes a U.

More data shrinks variance. Fitted on 1,000 points, the degree 9 curve has a test error close to the noise floor.

So the fix depends on the diagnosis. Training and test errors that are both high and close together point to bias: use a more flexible model or better features. A low training error with a much higher test error points to variance: get more data, or limit the model's flexibility.

In [ ]:
GRID = np.linspace(0.02, 0.98, 97)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, d, colour in zip(axes, (1, 3, 9), [PALETTE["cyan"], PALETTE["mint"], PALETTE["pink"]]):
    draw_fits(ax, _refits(d, 5, seed=5), colour, f"degree {d}: five training sets")
plt.tight_layout()
plt.show()



<font size="4"><b>Many training sets</b></font>

Write `refits(d, n, seed)`: make `r = np.random.default_rng(seed)`, then `n` times draw targets `truth(x) + r.normal(0, NOISE, 10)` at `x = np.linspace(0, 1, 10)` and fit a degree `d` polynomial. Return the list of coefficient arrays.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

A list comprehension that calls `r.normal` once per fit.

</details>

In [ ]:
# Exercise 4.1: running this cell checks your answer
def refits(d: int, n: int, seed: int) -> list[np.ndarray]:
    ...  # your code here



<font size="4"><b>Predictions on the grid</b></font>

Write `grid_preds(coefs)`: a 2D array with one row per fit and one column per point of `GRID`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Evaluate each set of coefficients on `GRID` with `np.polyval` and stack the rows.

</details>

In [ ]:
# Exercise 4.2: running this cell checks your answer
def grid_preds(coefs: list[np.ndarray]) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Squared bias</b></font>

Write `bias2(P)` for a predictions array from `grid_preds`: average the rows, compare with `truth(GRID)`, square and average.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`P.mean(axis=0)` is the average fit.

</details>

In [ ]:
# Exercise 4.3: running this cell checks your answer
def bias2(P: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>Variance</b></font>

Write `variance(P)`: the variance across fits at each grid point, averaged over the grid.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`P.var(axis=0)` gives one variance per grid point.

</details>

In [ ]:
# Exercise 4.4: running this cell checks your answer
def variance(P: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>The three models</b></font>

Using 500 training sets with seed 11 for each degree, make `bv`, a dictionary from each degree in `DEGREES` to `(bias2, variance)`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

For each degree, build the prediction matrix from your `refits` (500 fits, seed 11) and `grid_preds`, then apply both functions.

</details>

In [ ]:
# Exercise 4.5: running this cell checks your answer
bv = ...  # your answer



<font size="4"><b>Three parts of the error</b></font>

For degree 9, check the split of the test error. Make `P9 = grid_preds(refits(9, 500, 11))`, then draw fresh targets `Y_new = truth(GRID) + np.random.default_rng(99).normal(0, NOISE, P9.shape)` and store `mc_err`, the mean squared error of `P9` against `Y_new`. Store `parts`, the squared bias plus the variance plus the noise variance.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`mse(P9, Y_new)` averages over every fit and every grid point.

</details>

In [ ]:
# Exercise 4.6: running this cell checks your answer
P9 = ...  # your answer
Y_new = ...  # your answer
mc_err = ...  # your answer
parts = ...  # your answer



<font size="4"><b>The U</b></font>

Make arrays `bias_curve` and `var_curve` for degrees 0 to 9, each with 500 training sets and seed 11. The curves appear when the check passes.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Compute `grid_preds(refits(d, 500, 11))` once per degree inside a loop and keep both numbers.

</details>

In [ ]:
# Exercise 4.7: running this cell checks your answer
bias_curve, var_curve = ..., ...  # your answer



<font size="4"><b>The bottom of the U</b></font>

Store `u_bottom`, the degree where squared bias plus variance is smallest.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The expected error is bias plus variance plus the noise floor, and the floor is the same at every degree.

</details>

In [ ]:
# Exercise 4.8: running this cell checks your answer
u_bottom = ...  # your answer



<font size="4"><b>More data</b></font>

Fit a degree 9 curve to 1,000 training points: `r = np.random.default_rng(21)`, `x_big = r.uniform(0, 1, 1000)`, then `y_big = truth(x_big) + r.normal(0, NOISE, 1000)`. Store its test error as `err9_big`, and in `helps_most` the degree (1, 3 or 9) that 1,000 more examples would help most.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Draw `x_big` before `y_big`, so the random numbers come out in the same order.

</details>

In [ ]:
# Exercise 4.9: running this cell checks your answer
r_big = ...  # your answer
x_big = ...  # your answer
y_big = ...  # your answer
err9_big = ...  # your answer
helps_most = ...  # your answer



<font size="4"><b>Diagnose from two numbers</b></font>

Write `diagnose(train, test)` with this rule: return `"variance"` when the test error is more than twice the training error, `"bias"` when it is not and the training error is above 0.15, and `"fine"` otherwise.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

A chain of `if` and `elif`, checking the variance case first.

</details>

In [ ]:
# Exercise 4.10: running this cell checks your answer
def diagnose(train: float, test: float) -> str:
    ...  # your code here



## <font color="#d0309f">Regularization</font>

Look at the weights of the degree 9 fit. To pass through every point, some of them reach about 229 in size, with opposite signs, cancelling each other between the points. Large weights are what let the curve swing.

**Regularization** adds a charge for large weights to the training loss. The most common charge is $\lambda$ times the sum of the squared weights, where $\lambda$ (lambda) is a number we choose, the **regularization strength**:
$$\text{objective}(w) = \frac1n \sum_{i=1}^n \big(\phi(x_i)\cdot w - y_i\big)^2 + \lambda \sum_{j \ge 1} w_j^2.$$
The bias $w_0$ is usually left out of the penalty.

The penalty treats every weight alike, so the features should have similar sizes. Here the input is first rescaled to $u = 2x - 1$, which runs from $-1$ to $1$, and the features are $1, u, u^2, \dots, u^9$.

Setting the gradient of the objective to 0 gives the weights in closed form. With $X$ the matrix of features (one row per point) and $R$ the identity with its top left entry set to 0:
$$w = \Big(\tfrac1n X^\top X + \lambda R\Big)^{-1} \tfrac1n X^\top y.$$

With $\lambda = 0$ the curve threads every point. At $\lambda = 0.001$ the largest weight drops to about 3 and the curve is close to the wave. At $\lambda = 1$ the weights shrink toward 0 and the curve flattens: it underfits.

To set $\lambda$, fit at several values and score each on separate held-out points, called a **validation set**. Here that is the 10 points `VA_X`, `VA_Y`.

The penalty's gradient is $2\lambda w$, so each gradient step first shrinks every weight by a small fraction, then applies the usual push from the data. That is why it is also called **weight decay**. A second way to limit flexibility is **early stopping**: watch the validation error during training, and stop where it is lowest. Both carry over to neural networks unchanged.

In [ ]:
LAMBDAS = (0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
g = np.linspace(0, 1, 400)
axes[0].plot(g, truth(g), color=PALETTE["ink"], lw=1, ls=":")
for lam, colour in zip((0.0, 1e-3, 1.0), [PALETTE["pink"], PALETTE["mint"], PALETTE["cyan"]]):
    axes[0].plot(g, _f9(g) @ _ridge(lam), color=colour, label=f"λ = {lam:g}")
axes[0].scatter(TR_X, TR_Y, s=45, color=PALETTE["amber"], zorder=4)
axes[0].set_ylim(-2, 2); axes[0].legend(fontsize=8); axes[0].set_title("Three strengths")
for k, (lam, colour) in enumerate(zip((0.0, 1e-3, 1.0), [PALETTE["pink"], PALETTE["mint"], PALETTE["cyan"]])):
    axes[1].bar(np.arange(1, 10) + 0.27 * (k - 1), np.abs(_ridge(lam)[1:]), 0.27, color=colour, label=f"λ = {lam:g}")
axes[1].set_yscale("log"); axes[1].set_xlabel("power of u"); axes[1].set_title("Weight sizes"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()



<font size="4"><b>The penalty</b></font>

Write `penalty(w, lam)`: `lam` times the sum of the squared entries of `w`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`np.square` and `np.sum` do the work.

</details>

In [ ]:
# Exercise 5.1: running this cell checks your answer
def penalty(w, lam: float) -> float:
    return ...  # your code here



<font size="4"><b>Rescaled features</b></font>

Write `features9(x)`: the matrix with columns $1, u, u^2, \dots, u^9$ for $u = 2x - 1$, one row per entry of `x`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`np.vander(u, 10, increasing=True)` puts the powers in increasing order.

</details>

In [ ]:
# Exercise 5.2: running this cell checks your answer
def features9(x) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Fit with a penalty</b></font>

Write `ridge(lam, x, y)` with the closed form above. Add `1e-12 * np.eye(10)` inside the inverse so that $\lambda = 0$ still solves.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Build `R = np.eye(10)` and set `R[0, 0] = 0`. Use `np.linalg.solve(A, b)`, which is safer than inverting `A`.

</details>

In [ ]:
# Exercise 5.3: running this cell checks your answer
def ridge(lam: float, x=TR_X, y=TR_Y) -> np.ndarray:
    ...  # your code here



<font size="4"><b>The table</b></font>

Make `ridge_rows`, a dictionary from each value in `LAMBDAS` to `(train, val, biggest)`: the training error, the validation error and the largest absolute weight, bias excluded.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`features9(TR_X) @ w` gives the training predictions; slice `w[1:]` to leave out the bias.

</details>

In [ ]:
# Exercise 5.4: running this cell checks your answer
ridge_rows = ...  # your answer



<font size="4"><b>Pick lambda</b></font>

Store `best_lam`, the value with the lowest validation error, and `best_test`, its test error. Store `gain`, the test error of the plain degree 9 fit (`test_err[9]`) divided by `best_test`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`min` over `LAMBDAS` with a `key` function that reads the validation error from `ridge_rows`.

</details>

In [ ]:
# Exercise 5.5: running this cell checks your answer
best_lam = ...  # your answer
best_test = ...  # your answer
gain = ...  # your answer



<font size="4"><b>Weight decay</b></font>

Write `decay_step(w, g, lam, eta)`: one gradient step on loss plus penalty, where `g` is the gradient of the loss alone. Then store `shrink`, the factor that multiplies each weight before the data's push, for `eta = 0.1` and `lam = 0.01`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Add the penalty's slope $2\lambda w$ to the gradient in the step, then collect the terms in $w$.

</details>

In [ ]:
# Exercise 5.6: running this cell checks your answer
def decay_step(w, g, lam: float, eta: float):
    return ...  # your code here


shrink = ...  # your answer



<font size="4"><b>The same step in PyTorch</b></font>

`torch.optim.SGD` has a `weight_decay` argument that adds `weight_decay * w` to each gradient, so `weight_decay = 2 * lam` matches our penalty (it also decays the bias). Start from `w = 0.1` in all 10 entries (float64), take one SGD step with `lr=0.1` and `weight_decay=0.02` on the mean squared error of `features9(TR_X) @ w` against `TR_Y`, and store the result as `w_torch`. Store `w_manual`, the same step with your `decay_step` and the gradient `2 / n * X.T @ (X @ w - y)`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`opt.zero_grad()`, compute the loss, `loss.backward()`, `opt.step()`, then `w.detach().numpy()`.

</details>

In [ ]:
# Exercise 5.7: running this cell checks your answer
w_torch, w_manual = ..., ...  # your answer



<font size="4"><b>Stop early</b></font>

Train the unregularized degree 9 model by gradient descent: start from `w = np.zeros(10)` and take 5000 steps of size 0.3 with gradient `2 / n * X.T @ (X @ w - TR_Y)`, where `X = features9(TR_X)`. After each step record the training and validation errors in arrays `path_train` and `path_val`. Store `stop_step`, the index of the lowest validation error, and `stop_val`, its value. The curves appear when the check passes.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Precompute `features9(VA_X)` once outside the loop. `int(np.argmin(path_val))` gives the index.

</details>

In [ ]:
# Exercise 5.8: running this cell checks your answer
path_train, path_val, stop_step, stop_val = ..., ..., ..., ...  # your answer



## <font color="#0077ff">Honest evaluation</font>

Honest evaluation starts by cutting the data into three parts, before anything is trained. Of 1,000 labelled examples, 700 become the training set, 150 the validation set and 150 the **test set**.

The training set fits the weights. The validation set chooses everything else: $\lambda$, the number of hidden units, when to stop, which features to use. These choices are **hyperparameters**: settings picked by the person training the model, outside gradient descent. The test set is used once, at the end, to report how well the final model does.

Every time a set of examples guides a choice, the choice fits those particular examples a little, and a score on data that guided a choice comes out too high. Suppose a team tries 50 settings, scores each on 150 test examples, and reports the best. If all 50 are equally good, each 80 percent right on new data, luck alone spreads their scores, and the best of 50 is about 87 percent on average. That gap is **leakage**: information from the test set leaked into a decision.

How big should the test set be? A score on $n$ examples of a model that is right with probability $p$ has standard deviation $\sqrt{p(1-p)/n}$, and lands within about $1.96$ standard deviations of $p$ about 95 percent of the time. With 150 examples at 80 percent, that is about 6 points either way.

When data is scarce, **cross-validation** cuts the training data into, say, 5 parts called folds. It trains 5 times, each time holding out a different fold, and averages the 5 validation scores.

If you look at the test score and then change the model, the test set has become a validation set. Collect a new one before you report a final number.

In [ ]:
SETTINGS_LIST = [1, 2, 5, 10, 20, 50, 100, 200]

r_demo = np.random.default_rng(0)
demo_scores = r_demo.binomial(150, 0.8, 50) / 150
fig, ax = plt.subplots(figsize=(6.4, 3))
ax.bar(range(50), demo_scores, color=[PALETTE["pink"] if s == demo_scores.max() else PALETTE["violet"] for s in demo_scores])
ax.axhline(0.8, color=PALETTE["mint"], lw=2, label="true accuracy of every setting")
ax.set_ylim(0.65, 0.92); ax.set_xlabel("setting"); ax.set_ylabel("test score"); ax.legend(fontsize=8)
ax.set_title("50 equally good settings, 150 test examples")
plt.show()



<font size="4"><b>Three splits</b></font>

Write `split_indices(n, seed)`: shuffle `range(n)` with `np.random.default_rng(seed).permutation(n)` and return three arrays, the first 70 percent, the next 15 percent and the last 15 percent.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

The cut points are `int(0.7 * n)` and `int(0.85 * n)`.

</details>

In [ ]:
# Exercise 6.1: running this cell checks your answer
def split_indices(n: int, seed: int):
    ...  # your code here



<font size="4"><b>Which split does which job</b></font>

Make `job`, a dictionary mapping each task to `"train"`, `"validation"` or `"test"`: `"fit the weights"`, `"choose lambda"`, `"choose the number of hidden units"`, `"decide when to stop"` and `"report the final score"`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Only one task touches the test set, and it happens once.

</details>

In [ ]:
# Exercise 6.2: running this cell checks your answer
job = ...  # your answer



<font size="4"><b>Best of many</b></font>

Write `best_of(settings, n_test, acc, trials, seed)`: with `r = np.random.default_rng(seed)`, draw `r.binomial(n_test, acc, (trials, settings)) / n_test` (each row is one team's scores for all its settings), take the best score in each row, and return the mean of those bests. Store `leak50 = best_of(50, 150, 0.8, 20000, 7)`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`.max(axis=1).mean()`.

</details>

In [ ]:
# Exercise 6.3: running this cell checks your answer
def best_of(settings: int, n_test: int, acc: float, trials: int, seed: int) -> float:
    r = ...  # your answer
    return ...  # your code here


leak50 = ...  # your answer



<font size="4"><b>More settings, more optimism</b></font>

Make `optimism`, an array with `best_of(s, 150, 0.8, 20000, 7) - 0.8` for each `s` in `SETTINGS_LIST`. The curve appears when the check passes.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

A list comprehension inside `np.array`.

</details>

In [ ]:
# Exercise 6.4: running this cell checks your answer
optimism = ...  # your answer



<font size="4"><b>How much a test score wobbles</b></font>

For a model that is right 80 percent of the time and a test set of 150, store `sd150`, the standard deviation of its test score, and `half150`, 1.96 times that.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

$\sqrt{p(1-p)/n}$.

</details>

In [ ]:
# Exercise 6.5: running this cell checks your answer
sd150 = ...  # your answer
half150 = ...  # your answer



<font size="4"><b>A bigger test set</b></font>

How many test examples make the 95 percent half-width 2 points (0.02) for a model at 80 percent? Store the smallest whole number as `n_needed`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Solve $1.96\sqrt{p(1-p)/n} = 0.02$ for $n$, then round up with `math.ceil`.

</details>

In [ ]:
# Exercise 6.6: running this cell checks your answer
n_needed = ...  # your answer



<font size="4"><b>Folds</b></font>

Write `kfold(n, k)`: a list of `k` pairs `(train_idx, val_idx)`, where fold `f` holds out `np.arange(n)[f::k]` and trains on the rest.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`np.setdiff1d(np.arange(n), val_idx)` gives the rest.

</details>

In [ ]:
# Exercise 6.7: running this cell checks your answer
def kfold(n: int, k: int):
    ...  # your code here



<font size="4"><b>Cross-validate lambda</b></font>

With `kfold(10, 5)` on the 10 training points, write `cv_error(lam)`: fit `ridge(lam, ...)` on each fold's training part and average the 5 validation errors. Make `cv_errs`, a dictionary over `LAMBDAS`, and `cv_best`, the lambda it picks.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

For each fold, fit `ridge` on the training rows and score it on the held-out rows with `features9` and `mse`. Average the fold errors.

</details>

In [ ]:
# Exercise 6.8: running this cell checks your answer
def cv_error(lam: float) -> float:
    ...  # your code here


cv_errs = ...  # your answer
cv_best = ...  # your answer



<font size="4"><b>After a peek</b></font>

A team reports a test score, sees it is low, changes the features, and scores the new model on the same test set. Which is right? Store the letter as `peek`.

a. Report the new score; the test set was only used twice.
b. The test set has become a validation set; collect a new test set before reporting.
c. Report the average of the two scores.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Did the test set guide a choice?

</details>

In [ ]:
# Exercise 6.9: running this cell checks your answer
peek = ...  # your answer



<font size="4"><b>Weights or hyperparameters</b></font>

Make `kind`, a dictionary mapping each item to `"learned"` (set by gradient descent) or `"hyperparameter"`: `"hidden weights"`, `"output bias"`, `"lambda"`, `"step size"`, `"number of hidden units"`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Gradient descent changes only the numbers the loss has a slope for.

</details>

In [ ]:
# Exercise 6.10: running this cell checks your answer
kind = ...  # your answer



## <font color="#7b2ff7">Losses from likelihoods</font>

A **probabilistic model** gives a probability to each possible label, or to each possible value of the target. The **likelihood** of a model is the probability it gives to the data we observed.

Start with a coin flipped 3 times: heads, heads, tails. A model of the coin is one number $p$, its probability of heads, and the likelihood of the flips is $p \cdot p \cdot (1 - p)$. At $p = \tfrac12$ it is $\tfrac18$. It peaks at $p = \tfrac23$, at about 0.15. At $p = 0.9$ it is only about 0.08: a model that is too sure of itself pays for every surprise.

Choosing the model that makes the observed data most probable is **maximum likelihood**. For the coin it picks $\tfrac23$, the fraction of heads we saw.

With many examples, the likelihood is a product of many small probabilities. The log turns it into a sum, and maximizing the log likelihood is the same as minimizing the **negative log likelihood**: one term per example, the shape of a training loss. The log also keeps the numbers usable, since a product of thousands of probabilities is too small for a computer to store.

For regression, assume each target is the prediction plus noise from a bell curve, a **Gaussian**, with spread $\sigma$. Minus the log of its density at residual $r$ is
$$\frac{r^2}{2\sigma^2} + \log\big(\sigma\sqrt{2\pi}\big).$$
With $\sigma = 1$ that is $\tfrac12 r^2 + 0.92$. Apart from the constant, the cost is the squared residual: minimizing the squared loss is maximum likelihood under Gaussian noise.

For classification, treat each label as a biased coin. The sigmoid of the score gives the probability of $+1$, and one minus it the probability of $-1$. Minus the log of the probability given to the true label is the logistic loss from the last lesson.

So each loss carries an assumption. The squared loss assumes symmetric, bell-shaped noise, which is why one wild outlier can pull a regression line far off: under a bell curve that point is almost impossible, and the fit moves a long way to make it less so.

In [ ]:
FLIPS = "HHT"
p = np.linspace(0, 1, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(p, p * p * (1 - p), color=PALETTE["amber"])
for pp, colour in [(0.5, PALETTE["cyan"]), (2 / 3, PALETTE["mint"]), (0.9, PALETTE["pink"])]:
    axes[0].scatter([pp], [pp * pp * (1 - pp)], color=colour, s=70, zorder=3, label=f"p = {pp:.2f}")
axes[0].set_xlabel("probability of heads"); axes[0].set_ylabel("likelihood of H, H, T"); axes[0].legend(fontsize=8)
r = np.linspace(-3, 3, 300)
axes[1].plot(r, np.exp(-r ** 2 / 2) / math.sqrt(2 * math.pi), color=PALETTE["violet"], label="bell curve")
axes[1].plot(r, 0.5 * r ** 2 + 0.5 * math.log(2 * math.pi), color=PALETTE["cyan"], label="minus its log")
axes[1].set_ylim(0, 4); axes[1].set_xlabel("residual"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()



<font size="4"><b>The coin&#x27;s likelihood</b></font>

Write `coin_likelihood(p, flips)`: the product, over the letters of `flips`, of `p` for each `"H"` and `1 - p` for each `"T"`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Each flip contributes $p$ or $1 - p$, and `math.prod` multiplies a sequence.

</details>

In [ ]:
# Exercise 7.1: running this cell checks your answer
def coin_likelihood(p: float, flips: str) -> float:
    return ...  # your code here



<font size="4"><b>Three coins</b></font>

Make `coin_vals`, the likelihood of `FLIPS` at $p$ = 0.5, 2/3 and 0.9.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Your `coin_likelihood` in a list comprehension.

</details>

In [ ]:
# Exercise 7.2: running this cell checks your answer
coin_vals = ...  # your answer



<font size="4"><b>Maximum likelihood on a grid</b></font>

Over `P_GRID = np.linspace(0, 1, 1001)`, make the array `lik` of likelihoods and store `p_best`, the grid value with the largest likelihood.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`np.argmax` gives the index of the largest likelihood; read `P_GRID` at that index.

</details>

In [ ]:
# Exercise 7.3: running this cell checks your answer
P_GRID = np.linspace(0, 1, 1001)
lik = ...  # your answer
p_best = ...  # your answer



<font size="4"><b>Products underflow</b></font>

Store `prod_2000`, the product of 2,000 probabilities each equal to 0.4, computed with `np.prod`, and `logsum_2000`, the sum of their logs.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`np.full(2000, 0.4)` makes the array.

</details>

In [ ]:
# Exercise 7.4: running this cell checks your answer
prod_2000 = ...  # your answer
logsum_2000 = ...  # your answer



<font size="4"><b>The Gaussian cost</b></font>

Write `gauss_nll(r, sigma=1.0)`, minus the log of the Gaussian density at residual `r`. Use `math.log` and `math.pi` for the constant so that it also works on PyTorch tensors. Then make `nll_vals` at residuals 0, 1 and 2.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Take minus the log of the Gaussian density and simplify it into a squared term plus a constant.

</details>

In [ ]:
# Exercise 7.5: running this cell checks your answer
def gauss_nll(r, sigma: float = 1.0):
    return ...  # your code here


nll_vals = ...  # your answer



<font size="4"><b>Fit a line by likelihood</b></font>

In PyTorch, fit a line $a x + b$ to `TR_X`, `TR_Y` by minimizing the mean Gaussian negative log likelihood with your `gauss_nll`. Start `a` and `b` at 0 (float64), use `torch.optim.SGD` with `lr=0.5` for 3000 steps, and store `ab_nll = (a, b)` as floats. Store `ab_ls`, the least squares line from `np.polyfit(TR_X, TR_Y, 1)`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Make a 2-entry parameter tensor with `requires_grad=True` for the slope and intercept. The loss is the mean of your `gauss_nll` over the residuals.

</details>

In [ ]:
# Exercise 7.6: running this cell checks your answer
ab_nll, ab_ls = ..., ...  # your answer



<font size="4"><b>The label as a coin</b></font>

Write `prob_of_label(s, y)`: the probability the model gives to label `y` when the score is `s`, with the sigmoid for $+1$ and one minus it for $-1$. Then store `nll_s1`, minus the log of `prob_of_label(1.0, 1)`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

$\sigma(s) = 1 / (1 + e^{-s})$.

</details>

In [ ]:
# Exercise 7.7: running this cell checks your answer
def prob_of_label(s: float, y: int) -> float:
    p1 = ...  # your answer
    return ...  # your code here


nll_s1 = ...  # your answer



<font size="4"><b>Equal everywhere</b></font>

Over the scores `S_GRID` and both labels, store `max_gap`, the largest absolute difference between `-math.log(prob_of_label(s, y))` and the logistic loss `np.logaddexp(0, -y * s)`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

A `max` over a generator with two `for` clauses.

</details>

In [ ]:
# Exercise 7.8: running this cell checks your answer
S_GRID = np.linspace(-6, 6, 121)
max_gap = ...  # your answer



<font size="4"><b>One wild point</b></font>

Copy `TR_Y`, add 5 to its last entry, and fit a least squares line. Store `slope_clean` and `slope_outlier`, the slopes without and with the change.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`y_out = TR_Y.copy()`, then add 5 to `y_out[-1]`. The slope is the first coefficient from `np.polyfit`.

</details>

In [ ]:
# Exercise 7.9: running this cell checks your answer
y_out = ...  # your answer
slope_clean = ...  # your answer
slope_outlier = ...  # your answer



<font size="4"><b>Which loss, which assumption</b></font>

Make `loss_for`, a dictionary with two keys: `"gaussian noise"` and `"biased coin"`, each mapped to `"squared"` or `"logistic"`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The sigmoid gives the coin's bias.

</details>

In [ ]:
# Exercise 7.10: running this cell checks your answer
loss_for = ...  # your answer



## <font color="#e52e71">K-means by hand</font>

Suppose we have 8 points and no labels. All we know is where each point sits, and we ask whether they fall into groups. A group of nearby points is a **cluster**, and finding clusters without labels is **clustering**. It is our first example of **unsupervised learning**: learning from inputs alone. Clustering is used to group customers by how they shop, to group support tickets by topic before anyone labels them, and to shrink images by replacing similar colours with one.

**K-means** is the classic method. We choose the number of clusters $k$ in advance, here 2. Each cluster has a centre, a point that stands for the whole cluster, also called a **centroid**. The cost is the sum, over all points, of the squared distance from each point to its own centre. The squared distance between two points is the gap in the first coordinate squared plus the gap in the second coordinate squared.

K-means alternates two steps:

* **Assignment**: each point joins the centre nearest to it.
* **Update**: each centre moves to the mean of its points.

Start with the centres at $(3, 3)$ and $(2, 2)$. The point $(6, 5)$ is 13 from the first and 25 from the second, so it joins the first. The first centre takes 6 points and the second takes 2, with cost 88. The update moves the centres to $(5.5, 4.5)$ and $(1.5, 1.5)$, and the cost drops to 36. Assign again (cost 26), move again (cost 11), and the next assignment changes nothing, so K-means stops.

Each step can only lower the cost. Assignment gives each point its nearest centre, so no distance grows. The mean is the spot with the smallest total squared distance to a group, so the update cannot raise the cost either. There are finitely many ways to assign points to centres, so the method must stop.

In NumPy, broadcasting gives every point-to-centre distance at once: `P[:, None, :] - C[None, :, :]` has shape `(n, k, 2)`.

In [ ]:
KM_P = np.array([(3, 3), (2, 2), (1, 1), (2, 4), (6, 5), (7, 4), (8, 6), (7, 5)], dtype=float)
KM_C0 = np.array([(3, 3), (2, 2)], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(KM_P[:, 0], KM_P[:, 1], s=80, color=PALETTE["ink"])
axes[0].set_title("8 points, no labels"); axes[0].set_aspect("equal")
C_end, a_end, _ = _km(KM_P, KM_C0)
draw_clusters(axes[1], KM_P, C_end, a_end, "Where K-means ends")
plt.tight_layout()
plt.show()



<font size="4"><b>Squared distance</b></font>

Write `sqdist(a, b)` for two points given as arrays of any length.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Subtract, square and add up, with NumPy arrays.

</details>

In [ ]:
# Exercise 8.1: running this cell checks your answer
def sqdist(a, b) -> float:
    return ...  # your code here



<font size="4"><b>Every distance at once</b></font>

Write `dist_matrix(P, C)`: an array of shape `(n, k)` whose entry `[i, j]` is the squared distance from point `i` to centre `j`. Use broadcasting, with no Python loop.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Add axes with `None` so that every point lines up with every centre, then sum the squared differences over the last axis.

</details>

In [ ]:
# Exercise 8.2: running this cell checks your answer
def dist_matrix(P: np.ndarray, C: np.ndarray) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Assignment</b></font>

Write `assign(P, C)`: for each point, the index of its nearest centre. On a tie, take the lower index.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`np.argmin(..., axis=1)` already picks the first of equal values.

</details>

In [ ]:
# Exercise 8.3: running this cell checks your answer
def assign(P: np.ndarray, C: np.ndarray) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>The cost</b></font>

Write `kmeans_cost(P, C, a)`: the sum of squared distances from each point to the centre it is assigned to. Store `cost1`, the cost of the first assignment.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`C[a]` lines up each point with its own centre.

</details>

In [ ]:
# Exercise 8.4: running this cell checks your answer
def kmeans_cost(P: np.ndarray, C: np.ndarray, a: np.ndarray) -> float:
    return ...  # your code here


cost1 = ...  # your answer



<font size="4"><b>The update</b></font>

Write `update(P, a, C)`: move each centre to the mean of its points. A centre with no points stays where it is. Then store `C1`, the centres after the first update, and `cost1b`, the cost of the first assignment with those centres.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

For centre `j`, the points are `P[a == j]`. Build a new array; don't change `C` in place.

</details>

In [ ]:
# Exercise 8.5: running this cell checks your answer
def update(P: np.ndarray, a: np.ndarray, C: np.ndarray) -> np.ndarray:
    return ...  # your code here


C1 = ...  # your answer
cost1b = ...  # your answer



<font size="4"><b>By hand</b></font>

After the second assignment, the second centre owns $(3, 3)$, $(2, 2)$, $(1, 1)$ and $(2, 4)$. Work out by hand where it moves, and store it as `c2_hand`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Average the first coordinates, then the second ones.

</details>

In [ ]:
# Exercise 8.6: running this cell checks your answer
c2_hand = ...  # your answer



<font size="4"><b>The whole loop</b></font>

Write `kmeans(P, C)` returning `(C, a, history)`. Repeat: assign and append the cost to `history`; if the assignment equals the previous one, stop; otherwise update and append the cost again. Then store `km_history` for `KM_P` from `KM_C0`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Keep `prev = None` before the loop and compare with `np.array_equal(a, prev)`.

</details>

In [ ]:
# Exercise 8.7: running this cell checks your answer
def kmeans(P: np.ndarray, C: np.ndarray):
    ...  # your code here


km_history = ...  # your answer



<font size="4"><b>The mean is the best spot</b></font>

For the 4 left points `KM_P[:4]`, compute the total squared distance to each candidate centre in `CANDS` (a grid with spacing 0.1), and store `best_cand`, the candidate with the smallest total.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`dist_matrix(KM_P[:4], CANDS)` has one column per candidate; sum down the columns.

</details>

In [ ]:
# Exercise 8.8: running this cell checks your answer
g = np.round(np.arange(0, 5.01, 0.1), 1)
CANDS = np.array([(a, b) for a in g for b in g])
best_cand = ...  # your answer



<font size="4"><b>Why it stops</b></font>

Store `never_up`, whether `km_history` never increases from one entry to the next, and `n_assign`, the number of ways to assign 8 points to 2 centres.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`np.diff` gives the change between neighbouring costs. Each point has 2 choices.

</details>

In [ ]:
# Exercise 8.9: running this cell checks your answer
never_up = ...  # your answer
n_assign = ...  # your answer



<font size="4"><b>Fewer colours</b></font>

K-means works in any number of dimensions. Treat each pixel of the image `IMG` as a point with 3 colour values, run your `kmeans` from the centres `PIX[[0, 100, 300, 575]]`, and make `quant`, the image with every pixel replaced by its centre. The two images appear when the check passes.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`C[a]` gives each pixel's centre; `.reshape(IMG.shape)` turns the rows back into an image.

</details>

In [ ]:
# Exercise 8.10: running this cell checks your answer
yy, xx = np.mgrid[0:24, 0:24] / 23
IMG = np.stack([xx, 0.6 * yy + 0.2, 1 - xx * yy], axis=-1)
PIX = IMG.reshape(-1, 3)
C_img, a_img, _ = ..., ..., ...  # your answer
quant = ...  # your answer



## <font color="#00897b">K-means traps</font>

K-means always stops. It does not always stop at the best answer.

Take 12 points in 3 tight groups: lower left, upper left and right, and ask for 3 clusters. Start with one centre in each group, and K-means finishes with a cost of 6. Start with two centres in the right group and one on the left, between the two left groups, and it stops with a cost of 55: the single left centre takes both left groups, and the two right centres split the right group in half.

That is a **local minimum**: a solution that no single step can improve, even though a better one exists. The cure is cheap. Run K-means several times from different random starts and keep the run with the lowest cost. Better starts help too: a common rule spreads the first centres out, choosing each new one as the point farthest from the centres already picked.

The number of clusters is a second choice, and the cost alone cannot make it, because more clusters always lower the best cost. Plot the best cost against $k$ and look for the bend, the **elbow**, where the cost stops falling steeply.

K-means also assumes round clusters of similar size, because it measures plain distance to a centre. Two long, thin clusters side by side can defeat it.

In [ ]:
CL_LIST = [(1, 1), (2, 1), (1, 2), (2, 2), (1, 6), (2, 6), (1, 7), (2, 7), (7, 3), (8, 3), (7, 4), (8, 4)]
CL_P = np.array(CL_LIST, dtype=float)
GOOD_START = np.array([(2, 2), (1, 6), (8, 4)], dtype=float)
BAD_START = np.array([(7, 3), (7, 4), (1, 4)], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, start, name in [(axes[0], GOOD_START, "One centre per group"), (axes[1], BAD_START, "Two centres on the right")]:
    Cg, ag, hg = _km(CL_P, start)
    draw_clusters(ax, CL_P, Cg, ag, f"{name}: cost {hg[-1]:g}")
plt.tight_layout()
plt.show()



<font size="4"><b>The good start</b></font>

Run your `kmeans` on `CL_P` from `GOOD_START` and store the final cost as `cost_good`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The last entry of the history.

</details>

In [ ]:
# Exercise 9.1: running this cell checks your answer
cost_good = ...  # your answer



<font size="4"><b>The bad start</b></font>

Run from `BAD_START` and store `cost_bad` and `centres_bad`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`C, a, history = kmeans(...)`.

</details>

In [ ]:
# Exercise 9.2: running this cell checks your answer
centres_bad, _, hist_bad = ..., ..., ...  # your answer
cost_bad = ...  # your answer



<font size="4"><b>Stuck for good</b></font>

Take one more assignment step and one more update from `centres_bad`. Store `bad_fixed`, whether both the assignment and the centres stay the same.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Centres are stuck when assigning and updating again changes nothing. Check both with your `assign` and `update`; `np.allclose` compares centres.

</details>

In [ ]:
# Exercise 9.3: running this cell checks your answer
a_bad = ...  # your answer
bad_fixed = ...  # your answer



<font size="4"><b>Random restarts</b></font>

Make `rng_r = random.Random(4)` and run K-means 20 times, each from `np.array(rng_r.sample(CL_LIST, 3), dtype=float)`. Store the 20 final costs as `restart_costs`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Call `rng_r.sample` once per run, inside a list comprehension.

</details>

In [ ]:
# Exercise 9.4: running this cell checks your answer
rng_r = ...  # your answer
restart_costs = ...  # your answer



<font size="4"><b>Keep the best</b></font>

Store `best_restart`, the lowest of the 20 costs, and `n_stuck`, how many runs ended above it.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Count the runs whose final cost is above the best one, allowing `1e-9` for rounding.

</details>

In [ ]:
# Exercise 9.5: running this cell checks your answer
best_restart = ...  # your answer
n_stuck = ...  # your answer



<font size="4"><b>Spread-out starts</b></font>

Write `farthest_first(P, k, first)`: start with the point `P[first]`, then repeatedly add the point whose squared distance to its nearest chosen centre is largest. Store `spread_costs`, the final K-means cost from `farthest_first(CL_P, 3, i)` for every `i` in `range(12)`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

For each point, find its distance to the nearest chosen centre with your `dist_matrix`. The point with the largest such distance is the next centre.

</details>

In [ ]:
# Exercise 9.6: running this cell checks your answer
def farthest_first(P: np.ndarray, k: int, first: int) -> np.ndarray:
    ...  # your code here


spread_costs = ...  # your answer



<font size="4"><b>Cost against the number of clusters</b></font>

For each `k` from 1 to 6, run 30 restarts with `random.Random(10 + k)`, sampling `k` starting points each time, and keep the lowest final cost. Store the dictionary `best_by_k`. The elbow plot appears when the check passes.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Wrap your restart code in a function `best_cost(k)` that makes its own `random.Random(10 + k)`.

</details>

In [ ]:
# Exercise 9.7: running this cell checks your answer
best_by_k = ...  # your answer



<font size="4"><b>Read the elbow</b></font>

Store `elbow_k`, the smallest `k` for which going to `k + 1` lowers the best cost by less than 5 percent of the cost at `k = 1`. Store `cost_k12`, the cost when every one of the 12 points is its own centre.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Find the first `k` where one more cluster cuts the cost by less than 5% of the cost at $k = 1$. For the second part, start `kmeans` with every point as a centre.

</details>

In [ ]:
# Exercise 9.8: running this cell checks your answer
elbow_k = ...  # your answer
cost_k12 = ...  # your answer



<font size="4"><b>Two thin strips</b></font>

`STRIPS` holds two horizontal rows of 10 points, 1.5 apart. Store `cost_rows`, the K-means cost when each row is its own cluster (centres at the row means), and `cost_km`, the best final cost over 20 runs from `random.Random(0).sample(list(STRIPS), 2)` starts. Store `cuts_across`, whether K-means found a lower cost than the rows.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

For the rows, the assignment is `np.repeat([0, 1], 10)`; use your `update` and `kmeans_cost`. Make one `random.Random(0)` and sample from it 20 times.

</details>

In [ ]:
# Exercise 9.9: running this cell checks your answer
STRIPS = np.array([(x, 0.0) for x in range(10)] + [(x, 1.5) for x in range(10)])
rows = ...  # your answer
cost_rows = ...  # your answer
rng_s = ...  # your answer
cost_km = ...  # your answer
cuts_across = ...  # your answer



## <font color="#ff4d6d">Error by subgroup</font>

Test error is an average, and a good average can hide a group of examples where the model fails.

A ticket classifier is tested on 1,000 tickets and gets 4 percent wrong: 40 mistakes. 950 tickets came through the web form and 50 from the phone app. On the web tickets the error is 2 percent, 19 mistakes. That leaves 21 mistakes for the 50 app tickets, an error of 42 percent.

The average looked fine because app tickets are only 5 percent of the test set. A model trained mostly on long web tickets has seen few short app tickets, and their features look different.

A **subgroup** is any slice of the data defined by something other than the label: where the ticket came from, its language, the customer's region, the kind of device. Checking the error for each subgroup is a standard part of honest evaluation. When the groups are people, the stakes rise: a model that fails for those who write in a second language treats some people worse than others, and the overall error hides it.

Write the result as a table, one row per group, with its size, its mistakes and its error. The overall row is the weighted average of the others. Small groups are easy to miss and hard to measure: 50 tickets are enough to see a 42 percent error, and a group of 5 would tell you very little.

In [ ]:
r_sg = np.random.default_rng(0)
SOURCE = np.array(["web"] * 950 + ["app"] * 50)
WRONG = np.zeros(1000, dtype=bool)
WRONG[:19] = True
WRONG[950:971] = True
order = r_sg.permutation(1000)
SOURCE, WRONG = SOURCE[order], WRONG[order]

fig, ax = plt.subplots(figsize=(5.6, 3))
ax.bar(["overall", "web form", "phone app"], [WRONG.mean(), WRONG[SOURCE == "web"].mean(), WRONG[SOURCE == "app"].mean()],
       color=[PALETTE["violet"], PALETTE["cyan"], PALETTE["pink"]])
ax.set_ylabel("error"); ax.set_title("One average, two very different groups")
plt.show()



<font size="4"><b>Count the mistakes</b></font>

By hand, from the percentages, store `total_mistakes` (4 percent of 1,000), `web_mistakes` (2 percent of 950), and `app_err`, the error on the 50 app tickets.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The app mistakes are whatever the web tickets don't account for.

</details>

In [ ]:
# Exercise 10.1: running this cell checks your answer
total_mistakes = ...  # your answer
web_mistakes = ...  # your answer
app_err = ...  # your answer



<font size="4"><b>Errors from the arrays</b></font>

Write `group_table(groups, wrong)`: a dictionary from each group name to `(size, mistakes, error)`. Store `table` for `SOURCE` and `WRONG`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

For each name in `np.unique(groups)`, the mask `groups == name` picks its rows.

</details>

In [ ]:
# Exercise 10.2: running this cell checks your answer
def group_table(groups: np.ndarray, wrong: np.ndarray) -> dict:
    return ...  # your code here


table = ...  # your answer



<font size="4"><b>The overall row</b></font>

Store `overall`, the weighted average of the group errors in `table`, weighted by group size. Compare it with `WRONG.mean()`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Add up size times error over the rows, then divide by the total size.

</details>

In [ ]:
# Exercise 10.3: running this cell checks your answer
overall = ...  # your answer



<font size="4"><b>The hidden error</b></font>

Write `hidden_error(n, overall_err, n_big, big_err)`: the error on the smaller group, given the overall error on `n` examples and the error of the larger group of `n_big`. Round the mistake counts to whole numbers.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Turn the error rates into mistake counts first. The small group's mistakes are what the big group doesn't account for.

</details>

In [ ]:
# Exercise 10.4: running this cell checks your answer
def hidden_error(n: int, overall_err: float, n_big: int, big_err: float) -> float:
    return ...  # your code here



<font size="4"><b>Another population</b></font>

Overall error is 5 percent across 1,000 people: 900 in one group and 100 in the other. The larger group has an error of 2 percent. Store `small_err`, the smaller group's error, and `times`, how many times the larger group's error it is.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Call your `hidden_error`, then compare the result with the big group's error.

</details>

In [ ]:
# Exercise 10.5: running this cell checks your answer
small_err = ...  # your answer
times = ...  # your answer



<font size="4"><b>How sure is a small group&#x27;s number</b></font>

The measured error of a group of $n$ with true error $p$ has standard deviation $\sqrt{p(1-p)/n}$. For $p = 0.42$, store `sd50` for 50 tickets and `sd5` for 5.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The same formula as for test scores.

</details>

In [ ]:
# Exercise 10.6: running this cell checks your answer
sd50 = ...  # your answer
sd5 = ...  # your answer



<font size="4"><b>A group of 5 that looks perfect</b></font>

Store `p_zero5`, the probability that a group of 5 with true error 0.42 shows no mistakes at all, and `p_zero50`, the same for a group of 50.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Each ticket is right with probability 0.58, independently.

</details>

In [ ]:
# Exercise 10.7: running this cell checks your answer
p_zero5 = ...  # your answer
p_zero50 = ...  # your answer



<font size="4"><b>Simulate the wobble</b></font>

Draw 10,000 measured errors for a group of 5 and for a group of 50, with true error 0.42: `np.random.default_rng(3).binomial(n, 0.42, 10000) / n` (one generator, the group of 5 first). Store `sim_sd5` and `sim_sd50`, their standard deviations.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Make the generator once, call `binomial` twice, and use `.std()`.

</details>

In [ ]:
# Exercise 10.8: running this cell checks your answer
r3 = ...  # your answer
sim_sd5 = ...  # your answer
sim_sd50 = ...  # your answer



<font size="4"><b>Big enough to measure</b></font>

What is the smallest group size $n$ for which the standard deviation of a measured error of 0.42 is at most 0.05? Store it as `n_group`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Solve $\sqrt{p(1-p)/n} \le 0.05$ for $n$ and round up.

</details>

In [ ]:
# Exercise 10.9: running this cell checks your answer
n_group = ...  # your answer



<font size="4"><b>What to do next</b></font>

Which of these fixes come from this lesson? Store `fixes`, the set of letters.

a. More training tickets from the phone app.
b. Features that work for short tickets.
c. Drop the app tickets from the test set so the error looks better.
d. A test set with enough app tickets to measure their error.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

One option only hides the problem.

</details>

In [ ]:
# Exercise 10.10: running this cell checks your answer
fixes = ...  # your answer



## <font color="#1e88e5">The payoff</font>

Back to the four corners: amber at $(0, 0)$ and $(1, 1)$, cyan at $(0, 1)$ and $(1, 0)$. No straight line separates them.

The two hidden units compute their ramps. At the amber corners both are 0; at each cyan corner exactly one of them is 1. Together they fold the square along its diagonal, and in the plane of hidden values the line where they add up to $\tfrac12$ separates the colours.

As a network that is 2 inputs, 2 hidden units and 1 output, 9 weights. Below you'll build it in PyTorch with the weights set by hand, then train one with 8 hidden units from random weights.

One caution from the middle of this lesson. Many networks fit 4 points perfectly, and a perfect training score tells you nothing about new data. A network earns trust on a test set it never touched, checked group by group.

In [ ]:
HAND = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.ReLU(), torch.nn.Linear(2, 1)).double()
XOR_T = torch.tensor(XOR_X)
r_noisy = np.random.default_rng(1)
CORNER_OF = r_noisy.integers(0, 4, 400)
NOISY_X = XOR_X[CORNER_OF] + r_noisy.normal(0, 0.15, (400, 2))
NOISY_Y = XOR_Y[CORNER_OF]

fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.scatter(NOISY_X[:, 0], NOISY_X[:, 1], s=12, color=[CORNER_COLOURS[c] for c in CORNER_OF], alpha=0.7)
ax.set_aspect("equal"); ax.set_title("400 new points near the corners")
plt.show()



<font size="4"><b>Set the weights by hand</b></font>

Copy the hand weights into `HAND` inside `torch.no_grad()`. A `Linear` layer computes `x @ weight.T + bias`, so its `weight` has one row per unit: set `HAND[0].weight` to `U.T`, `HAND[0].bias` to `C`, `HAND[2].weight` to `V` as a 1 by 2 array, and `HAND[2].bias` to `[B]`. Store `hand_scores`, the four outputs on `XOR_T` as a flat NumPy array.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`HAND[0].weight.copy_(torch.tensor(U.T))` and the same pattern for the rest. `HAND(XOR_T).detach().numpy().ravel()` gives the scores.

</details>

In [ ]:
# Exercise 11.1: running this cell checks your answer
hand_scores = ...  # your answer



<font size="4"><b>Inside the model</b></font>

Store `hand_hidden`, the hidden values of `HAND` on `XOR_T` (run only the first two layers, `HAND[:2]`), as a NumPy array, and `hand_count`, its number of parameters.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Slicing an `nn.Sequential` gives a smaller one.

</details>

In [ ]:
# Exercise 11.2: running this cell checks your answer
hand_hidden = ...  # your answer
hand_count = ...  # your answer



<font size="4"><b>The fold over the square</b></font>

Make `fold_map`, the hand network's score at every point of the 101 by 101 grid `SQ` (shape `(101 * 101, 2)`), reshaped to `(101, 101)`. The map appears when the check passes.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Run `HAND` on the grid points as a tensor, then turn the scores back into a 101 by 101 array.

</details>

In [ ]:
# Exercise 11.3: running this cell checks your answer
gs = np.linspace(-0.2, 1.2, 101)
SQ = np.stack(np.meshgrid(gs, gs), -1).reshape(-1, 2)
fold_map = ...  # your answer



<font size="4"><b>Learn the units</b></font>

Call `torch.manual_seed(0)`, build `NET8 = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1)).double()`, and train it with `torch.optim.SGD(NET8.parameters(), lr=0.5)` for 2000 steps on the mean logistic loss over the four corners, `F.softplus(-y * s).mean()`. Store `net8_scores`, its four final scores, and `net8_ok`, whether all four signs are right.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`softplus(z)` is $\log(1 + e^z)$, so `softplus(-y * s)` is the logistic loss. The labels as a tensor: `torch.tensor(XOR_Y)`. Use `NET8(XOR_T).squeeze(1)` for the scores.

</details>

In [ ]:
# Exercise 11.4: running this cell checks your answer
net8_scores, net8_ok = ..., ...  # your answer



<font size="4"><b>Two perfect networks disagree</b></font>

The hand network and the second network from section 1 both fit the four corners perfectly. Store `disagree`, the fraction of points in `SQ` where their scores have different signs. Use `fold_map` for the first and compute the second with NumPy.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Write the second network's score from section 1 with your `relu` on the rows of `SQ`. `np.sign` tells you which side each point is on.

</details>

In [ ]:
# Exercise 11.5: running this cell checks your answer
s_second = ...  # your answer
disagree = ...  # your answer



<font size="4"><b>The centre of the square</b></font>

No training point sits at $(0.5, 0.5)$. Store `centre_signs`, a dictionary with the sign (+1 or -1) of the score at that point for `"hand"`, `"second"` and `"net8"`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Run each model on the single point. For the PyTorch models, `torch.tensor([[0.5, 0.5]])` is a batch of one; `.item()` gives the number.

</details>

In [ ]:
# Exercise 11.6: running this cell checks your answer
centre_signs = ...  # your answer



<font size="4"><b>A test set they never touched</b></font>

Store `acc_noisy`, a dictionary with the accuracy on `NOISY_X`, `NOISY_Y` of `"hand"`, `"second"` and `"net8"`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

For each model, compute the scores on `NOISY_X` and take the mean of `np.sign(scores) == NOISY_Y`.

</details>

In [ ]:
# Exercise 11.7: running this cell checks your answer
acc_noisy = ...  # your answer



<font size="4"><b>Group by group</b></font>

Check the hand network's error on the noisy points separately for each corner. Store `corner_err`, an array of 4 errors in the order of `XOR_X`, using `CORNER_OF` as the group label.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Your `group_table` works with integer groups too, or use a mask `CORNER_OF == c` for each corner.

</details>

In [ ]:
# Exercise 11.8: running this cell checks your answer
wrong_hand = ...  # your answer
corner_err = ...  # your answer



## <font color="#d84315">What you built</font>

* A hidden unit is a linear score passed through ReLU. Two of them fold the square so that one line splits it, and without the bend the network collapses to a single line.
* A network is layers of hidden units. You counted its weights, ran the forward pass, sent slopes back by hand and with autograd, and saw small networks get stuck where wider ones succeed.
* Training error rewards flexibility; test error on held-out data measures what you want. Bias and variance split the test error, and more data shrinks variance.
* A penalty of $\lambda \sum w_j^2$, chosen on a validation set, calms an overfit curve. Weight decay and early stopping are the same idea inside training.
* Train, validation and test sets each have one job. Choosing on the test set inflates its score.
* The squared loss is maximum likelihood under Gaussian noise, and the logistic loss is maximum likelihood for a coin whose bias is the sigmoid of the score.
* K-means alternates assignment and update, never raises its cost, and can stop in a local minimum, so restart it and spread the starts.
* An average error can hide a subgroup where the model fails. Measure each group, with enough examples to trust the number.

The next lesson takes a classifier from a good test score to a model that holds up after launch.



Run the cell below any time to see how far you are.

In [ ]:
progress()

<font size="2" color="#8a84b3">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</font>